### Loading different models for each and every case

In [1]:
pip show ultralytics

Name: ultralytics
Version: 8.4.33
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation and image classification.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /home/kunal-pg/miniconda3/envs/bnn/lib/python3.12/site-packages
Requires: matplotlib, numpy, opencv-python, pillow, polars, psutil, pyyaml, requests, scipy, torch, torchvision, ultralytics-thop
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
print(torch.cuda.device_count())  # Number of GPUs
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

2
GPU 0: NVIDIA L40
GPU 1: NVIDIA L40


In [2]:
import gdown
url = "https://drive.google.com/drive/folders/1yYtKpg4lmhyWATWLOF-Pa1c-5OGPoJeH"
url = "https://drive.google.com/drive/folders/1dFuncoGBNq_F2goptt4g9lIfixVI-GB0?usp=drive_link"
gdown.download(url,quiet = False)    


/home/kunal-pg/miniconda3/envs/bnn/lib/python3.12/site-packages/gdown/parse_url.py:48: UserWarning: You specified a Google Drive link that is not the correct link to download a file. You might want to try `--fuzzy` option or the following url: https://drive.google.com/uc?id=None
  warnings.warn(
Downloading...
From: https://drive.google.com/drive/folders/1dFuncoGBNq_F2goptt4g9lIfixVI-GB0?usp=drive_link
To: /home/kunal-pg/VLA_0/vla0/Bayesian/BNN/1dFuncoGBNq_F2goptt4g9lIfixVI-GB0?usp=drive_link
1.36MB [00:01, 1.13MB/s]


'1dFuncoGBNq_F2goptt4g9lIfixVI-GB0?usp=drive_link'

In [3]:
from ultralytics import YOLO
# Model for classification 
model_cls = YOLO("yolo11s-cls.pt")
# Model for Oriented_detection
model_obb = YOLO("yolo11s-obb.pt")
# Model for Pose/Keypoint Detection
model_pose = YOLO("yolo11s-pose.pt")
# Model for Instance Segementation
model_seg = YOLO("yolo11s-seg.pt")
# Model for Detection 
model_det = YOLO("yolo11s.pt")

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO
model_det = YOLO("yolo11s.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/kunal-pg/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from pathlib import Path
import yaml
from ultralytics.utils import ASSETS_URL
from ultralytics.utils.downloads import download

# Download labels
segments = True  # segment or box labels
dir = "/home/kunal-pg/VLA_0/vla0/Bayesian/BNN"  # dataset root dir
# urls = [ASSETS_URL + ("/coco2017labels-segments.zip" if segments else "/coco2017labels.zip")]  # labels
# urls = [ASSETS_URL]
# download(urls, dir=dir)
# Download data
urls = [
    "http://images.cocodataset.org/zips/train2017.zip",  # 19G, 118k images
    "http://images.cocodataset.org/zips/val2017.zip",  # 1G, 5k images
    "http://images.cocodataset.org/zips/test2017.zip",  # 7G, 41k images (optional)
]
download(urls, dir=dir, threads=3)

Unzipping /home/kunal-pg/VLA_0/vla0/Bayesian/BNN/val2017.zip to /home/kunal-pg/VLA_0/vla0/Bayesian/BNN/val2017...: 100% ━━━━━━━━━━━━ 5001/5001 3.7Kfiles/s 1.4s 1:36<29:08
Unzipping /home/kunal-pg/VLA_0/vla0/Bayesian/BNN/test2017.zip to /home/kunal-pg/VLA_0/vla0/Bayesian/BNN/test2017...: 100% ━━━━━━━━━━━━ 40671/40671 3.7Kfiles/s 10.9s<0.0s
Unzipping /home/kunal-pg/VLA_0/vla0/Bayesian/BNN/train2017.zip to /home/kunal-pg/VLA_0/vla0/Bayesian/BNN/train2017...: 100% ━━━━━━━━━━━━ 118288/118288 664.4files/s 2:580.0ss7


**Model Architecture**

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# BAYESIAN LINEAR LAYER
# =========================================================
class BayesianLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, prior_std: float = 1.0):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.prior_std = prior_std

        # Mean parameters
        self.weight_mean = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias_mean = nn.Parameter(torch.randn(out_features) * 0.1)

        # Log variance (log σ²)
        self.weight_logvar = nn.Parameter(torch.randn(out_features, in_features) * 0.1 - 5)
        self.bias_logvar = nn.Parameter(torch.randn(out_features) * 0.1 - 5)

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        if sample:
            weight_std = torch.exp(0.5 * self.weight_logvar)
            bias_std = torch.exp(0.5 * self.bias_logvar)

            weight = self.weight_mean + weight_std * torch.randn_like(weight_std)
            bias = self.bias_mean + bias_std * torch.randn_like(bias_std)
        else:
            weight = self.weight_mean
            bias = self.bias_mean

        return F.linear(x, weight, bias)

    def get_kl_divergence(self) -> torch.Tensor:
        """
        KL divergence between:
        q(w|θ) = N(μ, σ²)
        p(w)   = N(0, prior_std²)
        """
        weight_var = torch.exp(self.weight_logvar)
        bias_var = torch.exp(self.bias_logvar)

        prior_var = torch.tensor(self.prior_std ** 2, device=weight_var.device)

        kl_weight = 0.5 * torch.sum(
            (weight_var + self.weight_mean**2) / prior_var
            - 1.0
            + torch.log(prior_var)
            - self.weight_logvar
        )

        kl_bias = 0.5 * torch.sum(
            (bias_var + self.bias_mean**2) / prior_var
            - 1.0
            + torch.log(prior_var)
            - self.bias_logvar
        )

        return kl_weight + kl_bias


# =========================================================
# BAYESIAN HEAD
# =========================================================
class BayesianDetectionHead(nn.Module):
    def __init__(self, input_channels: int = 512):
        super().__init__()

        self.fc1 = BayesianLinear(input_channels, 256)
        self.fc2 = BayesianLinear(256, 128)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        x = self.fc1(x, sample)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x, sample)
        x = self.relu(x)
        x = self.dropout(x)

        return x

    def get_kl_divergence(self) -> torch.Tensor:
        return self.fc1.get_kl_divergence() + self.fc2.get_kl_divergence()

# =========================================================
# NEW BAYESIAN HEAD FOR THE SEGMENTATION TASK SPECIFICALLY 
# =========================================================

class BayesianSegmentationHead(nn.Module):
    def __init__(self, in_channels: int = 512):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 128, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        # We don't use nn.Dropout2d here because we want manual control via F.dropout2d
        
        self.conv2 = nn.Conv2d(128, 1, kernel_size=1)
        self.upsample = nn.Upsample(size=(640, 640), mode='bilinear', align_corners=False)

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        x = self.conv1(x)
        x = self.relu(x)
        
        # ---> THE BAYESIAN INJECTION <---
        # If sample=True, randomly drop 30% of the 2D feature maps.
        # This creates the variance needed for Segmentation Uncertainty!
        x = F.dropout2d(x, p=0.3, training=sample)
        
        x = self.conv2(x)
        x = self.upsample(x)
        return x

    def get_kl_divergence(self) -> torch.Tensor:
        # MC Dropout mathematically approximates Bayesian Inference without an explicit KL loss term.
        # We return 0 so your B_YOLOTrainer's kl_div summation doesn't crash!
        return torch.tensor(0.0, device=next(self.parameters()).device)
    
# =========================================================
# B-YOLO MODEL
# =========================================================
class B_YOLO(nn.Module):
    def __init__(self, backbone: nn.Module, num_classes: int = 80):
        super().__init__()

        self.backbone = backbone
        self.num_classes = num_classes

        # Freeze backbone properly ✔️
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.bayesian_head = BayesianDetectionHead(512)

        # Task heads
        self.detection_head = nn.Linear(128, 4 + num_classes)
        # self.segmentation_head = nn.Linear(128, 256)
        # Adding a new segementation head 
        # self.segmentation_head= nn.Sequential(
        #     nn.Conv2d(256, 128, kernel_size=3, padding=1), 
        #     nn.ReLU(),
        #     nn.Conv2d(128, 1, kernel_size=1), # 1 channel for the binary mask
        #     nn.Upsample(size=(640, 640), mode='bilinear', align_corners=False)
        # )
        self.segmentation_head = BayesianSegmentationHead(in_channels=512)
        self.pose_head = nn.Linear(128, 17 * 2)
        self.oriented_head = nn.Linear(128, 5 + num_classes)
        self.classification_head = nn.Linear(128, num_classes)

        self.current_task = None

    def set_task(self, task: str):
        valid_tasks = ['detection', 'segmentation', 'pose', 'oriented', 'classification']
        if task not in valid_tasks:
            raise ValueError(f"Task must be one of {valid_tasks}")
        self.current_task = task

    def forward(
        self,
        x: torch.Tensor,
        sample: bool = True,
        num_mc_samples: int = 1,
        return_uncertainty: bool = True
    ) -> dict:
        """
        Forward pass with optional uncertainty estimation.

        Args:
            x: Input tensor
            sample: Whether to sample from Bayesian layers
            num_mc_samples: Number of MC samples for uncertainty (if > 1)
            return_uncertainty: Whether to compute and return uncertainty

        Returns:
            dict with 'predictions' and 'uncertainty' keys
        """
        # Backbone (with no grad)
        with torch.no_grad():
            features = self.backbone(x)

        # Flatten
        if features.dim() == 4 and getattr(self, 'current_task', None) != 'segmentation':
            features = F.adaptive_avg_pool2d(features, 1)
            features = features.view(features.size(0), -1)
            
        # special forward for the segmentation task only
        if getattr(self, 'current_task', None) == 'segmentation':
            
            if return_uncertainty and num_mc_samples > 1:
                predictions_list = []
                for _ in range(num_mc_samples):
                    # Pass the 4D features to the Bayesian Spatial Head
                    # 'sample=True' triggers the 2D MC Dropout!
                    pred = self.segmentation_head(features, sample=sample)
                    predictions_list.append(pred)

                predictions_stacked = torch.stack(predictions_list, dim=0)
                mean_predictions = predictions_stacked.mean(dim=0)
                uncertainty = predictions_stacked.var(dim=0)

                return {
                    'predictions': mean_predictions,
                    'uncertainty': uncertainty,
                    'num_mc_samples': num_mc_samples
                }
            else:
                # Single forward pass
                predictions = self.segmentation_head(features, sample=sample)
                return {
                    'predictions': predictions,
                    'uncertainty': torch.zeros_like(predictions),
                    'num_mc_samples': 1
                }

        # Compute predictions with uncertainty via MC sampling
        if return_uncertainty and num_mc_samples > 1:
            predictions_list = []

            for _ in range(num_mc_samples):
                # Bayesian head with stochastic sampling
                z = self.bayesian_head(features, sample=sample)

                # Task-specific output
                if self.current_task == 'detection':
                    pred = self.detection_head(z)
                elif self.current_task == 'pose':
                    pred = self.pose_head(z)
                elif self.current_task == 'oriented':
                    pred = self.oriented_head(z)
                elif self.current_task == 'classification':
                    pred = self.classification_head(z)
                else:
                    raise ValueError("Task not set")

                predictions_list.append(pred)

            # Stack predictions: (num_mc_samples, batch_size, num_outputs)
            predictions_stacked = torch.stack(predictions_list, dim=0)

            # Compute mean and variance across MC samples
            mean_predictions = predictions_stacked.mean(dim=0)
            uncertainty = predictions_stacked.var(dim=0)

            return {
                'predictions': mean_predictions,
                'uncertainty': uncertainty,
                'num_mc_samples': num_mc_samples
            }
        else:
            # Single forward pass (no uncertainty computation)
            x = self.bayesian_head(features, sample=sample)

            # Task-specific output
            if self.current_task == 'detection':
                predictions = self.detection_head(x)
            elif self.current_task == 'pose':
                predictions = self.pose_head(x)
            elif self.current_task == 'oriented':
                predictions = self.oriented_head(x)
            elif self.current_task == 'classification':
                predictions = self.classification_head(x)
            else:
                raise ValueError("Task not set")

            return {
                'predictions': predictions,
                'uncertainty': torch.zeros_like(predictions),
                'num_mc_samples': 1
            }

    def predict_with_uncertainty(self, x: torch.Tensor, T: int = 10):
        """
        Monte Carlo inference with detailed uncertainty estimates.
        """
        self.eval()

        preds = []
        with torch.no_grad():
            for _ in range(T):
                output = self.forward(x, sample=True, num_mc_samples=1, return_uncertainty=False)
                preds.append(output['predictions'])

        preds = torch.stack(preds)  # (T, B, D)

        if self.current_task == 'classification':
            probs = torch.softmax(preds, dim=-1)
            return {
                'mean': probs.mean(0),
                'uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'detection':
            bbox = preds[..., :4]
            logits = preds[..., 4:]

            probs = torch.softmax(logits, dim=-1)

            return {
                'bbox_mean': bbox.mean(0),
                'bbox_uncertainty': bbox.var(0),
                'class_probs_mean': probs.mean(0),
                'class_uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'oriented':
            bbox = preds[..., :4]
            angle = preds[..., 4:5]
            logits = preds[..., 5:]

            probs = torch.softmax(logits, dim=-1)

            return {
                'bbox_mean': bbox.mean(0),
                'bbox_uncertainty': bbox.var(0),
                'angle_mean': angle.mean(0),
                'angle_uncertainty': angle.var(0),
                'class_probs_mean': probs.mean(0),
                'class_uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'segmentation':
            probs = torch.sigmoid(preds)
            return {
                'mean': probs.mean(0),
                'uncertainty': probs.var(0)
            }

        elif self.current_task == 'pose':
            return {
                'mean': preds.mean(0),
                'uncertainty': preds.var(0)
            }

        else:
            raise ValueError("Unknown task")

    def get_kl_divergence(self):
        return self.bayesian_head.get_kl_divergence()


**Task-Specific Loss Functions**

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# TASK-SPECIFIC LOSS
# =========================================================
class TaskSpecificLoss(nn.Module):
    def __init__(self, task: str, num_classes: int = 80):
        super().__init__()
        self.task = task
        self.num_classes = num_classes

    def forward(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
        kl_divergence: torch.Tensor = None,
        kl_weight: float = 0.01
    ) -> torch.Tensor:

        if self.task == 'detection':
            task_loss = self._detection_loss(predictions, targets)

        elif self.task == 'segmentation':
            task_loss = self._segmentation_loss(predictions, targets)

        elif self.task == 'pose':
            task_loss = self._pose_loss(predictions, targets)

        elif self.task == 'oriented':
            task_loss = self._oriented_loss(predictions, targets)

        elif self.task == 'classification':
            task_loss = self._classification_loss(predictions, targets)

        else:
            raise ValueError(f"Unknown task: {self.task}")

        if kl_divergence is not None:
            task_loss = task_loss + kl_weight * kl_divergence

        return task_loss

    # =====================================================
    # DETECTION
    # =====================================================
    # def _detection_loss(self, predictions, targets):
    #     if isinstance(targets, list):
    #         parsed_boxes = []
            
    #         # Iterate through each image in the batch
    #         for img_annotations in targets:
    #             # Iterate through each bounding box in the image
    #             for ann in img_annotations:
    #                 # COCO bounding box format is [x_min, y_min, width, height]
    #                 bbox = ann['bbox']
                    
    #                 # Ensure the bbox is a tensor and on the correct device
    #                 if not isinstance(bbox, torch.Tensor):
    #                     bbox = torch.tensor(bbox, device=predictions.device, dtype=torch.float32)
                        
    #                 parsed_boxes.append(bbox)
            
    #         # Stack all bounding boxes into a single tensor [Num_Total_Boxes, 4]
    #         if len(parsed_boxes) > 0:
    #             targets = torch.stack(parsed_boxes)
    #         else:
    #             # Fallback if there are no objects in any image in the batch
    #             targets = torch.zeros((0, 4), device=predictions.device, dtype=torch.float32)
    #     bbox_pred = predictions[:, :4]
    #     bbox_target = targets[:, :4]

    #     bbox_loss = F.smooth_l1_loss(bbox_pred, bbox_target)

    #     class_pred = predictions[:, 4:]

    #     if targets.dim() == 2:
    #         class_target = targets[:, 4:]
    #         if class_target.shape[1] == self.num_classes:
    #             class_target = class_target.argmax(dim=1)
    #         else:
    #             raise ValueError("Invalid class target shape")
    #     else:
    #         raise ValueError("Targets must be 2D")

    #     class_loss = F.cross_entropy(class_pred, class_target)

    #     return bbox_loss * 0.5 + class_loss
    
    
    # =====================================================
    # New Detection Loss Function
    # =====================================================
    def _detection_loss(self, predictions, targets):
        # predictions shape is expected to be: [batch_size, num_predictions, 4 + num_classes]
        batch_size = predictions.size(0)
        
        total_bbox_loss = 0.0
        total_class_loss = 0.0
        objects_found = 0

        # Loop through the batch image by image
        for i in range(batch_size):
            # 1. Get predictions for THIS specific image
            pred_for_image = predictions[i]
            if pred_for_image.dim() == 1:
                num_features = 4 + self.num_classes
                # .view(-1, num_features) automatically calculates the correct number of rows
                pred_for_image = pred_for_image.view(-1, num_features)
            pred_boxes = pred_for_image[:, :4]
            pred_classes = pred_for_image[:, 4:]

            # 2. Get COCO ground truth targets for THIS specific image
            img_targets = targets[i]

            # If there are no objects in this image, skip calculating loss for it
            if len(img_targets) == 0:
                continue

            # 3. Parse COCO dictionaries for this image
            gt_boxes = []
            gt_classes = []
            
            for ann in img_targets:
                # Extract Bounding Box
                bbox = ann['bbox']
                if not isinstance(bbox, torch.Tensor):
                    bbox = torch.tensor(bbox, device=predictions.device, dtype=torch.float32)
                gt_boxes.append(bbox)
                
                # Extract Class Label (category_id)
                cat_id = int(ann['category_id'])
                # Safe-guard: COCO category IDs sometimes exceed standard ranges. 
                # Clamp it to your num_classes to prevent index out-of-bounds crashes.
                if cat_id >= self.num_classes:
                    cat_id = self.num_classes - 1 
                gt_classes.append(cat_id)

            gt_boxes = torch.stack(gt_boxes) # Shape: [num_gt_objects, 4]
            gt_classes = torch.tensor(gt_classes, device=predictions.device, dtype=torch.long) # Shape: [num_gt_objects]
            
            objects_found += gt_boxes.size(0)

            # ==========================================================
            # 4. NAIVE ASSIGNMENT (Matching Predictions to Ground Truth)
            # ==========================================================
            # Calculate the L1 distance between all predictions and all ground truths
            distances = torch.cdist(pred_boxes, gt_boxes, p=1) 
            
            # For each ground truth box, find the index of the closest prediction
            best_pred_indices = torch.argmin(distances, dim=0)

            # 5. Extract ONLY the matched predictions
            matched_pred_boxes = pred_boxes[best_pred_indices]
            matched_pred_classes = pred_classes[best_pred_indices]

            # ==========================================================
            # 6. Calculate Losses for this image
            # ==========================================================
            image_bbox_loss = F.smooth_l1_loss(matched_pred_boxes, gt_boxes)
            
            # cross_entropy automatically handles integer target indices (no argmax needed)
            image_class_loss = F.cross_entropy(matched_pred_classes, gt_classes)

            total_bbox_loss += image_bbox_loss
            total_class_loss += image_class_loss

        # 7. Average and combine the losses across the batch
        if objects_found > 0:
            # We divide by the number of objects, not the batch size, to normalize the loss
            avg_bbox_loss = total_bbox_loss / objects_found
            avg_class_loss = total_class_loss / objects_found
            
            # Combine them (bbox_loss is scaled by 0.5 as in your original code)
            return (avg_bbox_loss * 0.5) + avg_class_loss
        else:
            # Fallback if no objects are present in the entire batch:
            # Multiply by 0.0 to return a 0 loss, but keep the computation graph connected!
            return predictions.sum() * 0.0

    # =====================================================
    # SEGMENTATION
    # =====================================================
    def _segmentation_loss(self, predictions, targets):
        bce = F.binary_cross_entropy_with_logits(predictions, targets)

        probs = torch.sigmoid(predictions)

        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)

        dice = 1 - ((2 * intersection + 1e-8) / (union + 1e-8))
        dice = dice.mean()

        return 0.5 * bce + 0.5 * dice

    # =====================================================
    # POSE
    # =====================================================
    # def _pose_loss(self, predictions, targets):
    #     mse = F.mse_loss(predictions, targets)
    #     smooth = F.smooth_l1_loss(predictions, targets)
    #     return 0.7 * mse + 0.3 * smooth
    # =====================================================
    # NEW POSE LOSS
    # =====================================================
    def _pose_loss(self, predictions, targets):
        batch_size = predictions.size(0)
        total_pose_loss = 0.0
        objects_found = 0

        for i in range(batch_size):
            pred_for_image = predictions[i]
            if pred_for_image.dim() == 1:
                pred_for_image = pred_for_image.view(-1, 34)

            # targets[i] is now a LIST of your JSON dictionaries!
            img_targets = targets[i]
            if len(img_targets) == 0:
                continue

            gt_poses = []
            for ann in img_targets:
                # Extract the 51-length array from your JSON
                if 'keypoints' not in ann:
                    continue
                    
                kpts = ann['keypoints']
                if not isinstance(kpts, torch.Tensor):
                    kpts = torch.tensor(kpts, device=predictions.device, dtype=torch.float32)
                
                # Strip out the 3rd visibility values (0, 1, or 2) from your JSON.
                # This leaves exactly 34 values (17 X's and 17 Y's)
                xy_indices = [j for j in range(51) if j % 3 != 2]
                kpts_xy = kpts[xy_indices]
                
                gt_poses.append(kpts_xy)

            if len(gt_poses) == 0:
                continue

            gt_poses = torch.stack(gt_poses)
            objects_found += gt_poses.size(0)

            # Match predictions to the ground truth poses
            distances = torch.cdist(pred_for_image, gt_poses, p=2) 
            best_pred_indices = torch.argmin(distances, dim=0)
            matched_pred_poses = pred_for_image[best_pred_indices]

            # Calculate Loss
            image_pose_loss = F.mse_loss(matched_pred_poses, gt_poses)
            total_pose_loss += image_pose_loss

        if objects_found > 0:
            return total_pose_loss / objects_found
        else:
            return predictions.sum() * 0.0

    # =====================================================
    # ORIENTED DETECTION
    # =====================================================
    def _oriented_loss(self, predictions, targets):
        bbox_loss = F.smooth_l1_loss(predictions[:, :4], targets[:, :4])

        angle_pred = predictions[:, 4]
        angle_target = targets[:, 4]

        angle_diff = torch.atan2(
            torch.sin(angle_pred - angle_target),
            torch.cos(angle_pred - angle_target)
        )

        angle_loss = torch.mean(angle_diff ** 2)

        class_pred = predictions[:, 5:]

        class_target = targets[:, 5:]
        if class_target.shape[1] == self.num_classes:
            class_target = class_target.argmax(dim=1)

        class_loss = F.cross_entropy(class_pred, class_target)

        return 0.4 * bbox_loss + 0.3 * angle_loss + 0.3 * class_loss

    # =====================================================
    # CLASSIFICATION
    # =====================================================
    # def _classification_loss(self, predictions, targets):
    #     if targets.dim() == 2:
    #         targets = targets.argmax(dim=1)

    #     return F.cross_entropy(predictions, targets)
    def _classification_loss(self, predictions, targets):
        # 1. Convert tuple/list to PyTorch Tensor and move to correct device
        if isinstance(targets, (list, tuple)):
            # If targets is a list of tensors, stack them. 
            # If it's a list of raw numbers (ints/floats), tensor() handles it.
            if isinstance(targets[0], torch.Tensor):
                targets = torch.stack(targets).to(predictions.device)
            else:
                targets = torch.tensor(targets, device=predictions.device)

        # Ensure targets are long integers (required by cross_entropy)
        targets = targets.long()

        # 2. Check dimensions (One-Hot Encoded vs Class Indices)
        if targets.dim() == 2:
            targets = targets.argmax(dim=1)

        # 3. Calculate Loss
        return F.cross_entropy(predictions, targets)


# =========================================================
# EWC REGULARIZER
# =========================================================
class EWCRegularizer(nn.Module):
    def __init__(self, model: nn.Module, lambda_ewc: float = 0.4):
        super().__init__()
        self.model = model
        self.lambda_ewc = lambda_ewc

        self.fisher_information = {}
        self.previous_weights = {}

    def compute_fisher_information(
        self,
        dataloader,
        loss_fn,
        device
    ):
        self.model.eval()

        fisher = {
            name: torch.zeros_like(param, device=device)
            for name, param in self.model.named_parameters()
            if param.requires_grad
        }

        total_samples = 0

        for images, targets in dataloader:
            images = images.to(device)
            targets = targets.to(device)

            self.model.zero_grad()

            outputs = self.model(images, sample=False)
            loss = loss_fn(outputs, targets)

            loss.backward()

            for name, param in self.model.named_parameters():
                if param.grad is not None:
                    fisher[name] += (param.grad.detach() ** 2)

            total_samples += 1

        for name in fisher:
            fisher[name] /= max(total_samples, 1)
            fisher[name] += 1e-8  # stability

        self.fisher_information = fisher
        self._save_previous_weights()

    def _save_previous_weights(self):
        self.previous_weights = {
            name: param.detach().clone()
            for name, param in self.model.named_parameters()
            if param.requires_grad
        }

    def get_ewc_loss(self):
        if not self.fisher_information:
            return torch.tensor(0.0, device=next(self.model.parameters()).device)

        loss = torch.zeros(1, device=next(self.model.parameters()).device)

        for name, param in self.model.named_parameters():
            if name in self.fisher_information:
                fisher = self.fisher_information[name]
                prev = self.previous_weights[name]

                loss += torch.sum(fisher * (param - prev) ** 2)

        return (self.lambda_ewc / 2.0) * loss

**Training Loop with Best Model Saving**

In [5]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import os
import cv2
from pathlib import Path
import numpy as np
from tqdm import tqdm
from collections import defaultdict



# class B_YOLOTrainer:
#     def __init__(
#         self,
#         model,
#         tasks,
#         device=None,
#         checkpoint_dir='./checkpoints'
#     ):
#         self.model = model
#         self.tasks = tasks
#         self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#         self.checkpoint_dir = Path(checkpoint_dir)
#         self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

#         self.ewc_regularizer = EWCRegularizer(model)

#         self.best_metrics = defaultdict(lambda: {'score': 0.0, 'model_path': None})
#         self.training_history = defaultdict(list)

#     # =====================================================
#     # MAIN TRAINING LOOP
#     # =====================================================
#     def train_sequential_tasks(
#         self,
#         task_dataloaders,
#         num_epochs_per_task=50,
#         learning_rate=1e-3,
#         kl_weight=0.01,
#         lambda_ewc=0.4,
#         mc_samples_train=1,
#         mc_samples_val=3
#     ):
#         self.model.to(self.device)

#         for task_idx, task in enumerate(self.tasks):

#             print(f"\n{'='*60}")
#             print(f"Task {task_idx+1}/{len(self.tasks)}: {task.upper()}")
#             print(f"{'='*60}")

#             self.model.set_task(task)

#             loss_fn = TaskSpecificLoss(task, self.model.num_classes)

#             optimizer = optim.Adam(
#                 filter(lambda p: p.requires_grad, self.model.parameters()),
#                 lr=learning_rate
#             )

#             scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#                 optimizer, mode='max', factor=0.5, patience=5
#             )

#             train_loader = task_dataloaders[task]['train']
#             val_loader = task_dataloaders[task]['val']

#             best_val_metric = -float('inf')
#             patience_counter = 0
#             patience = 10

#             for epoch in range(num_epochs_per_task):

#                 train_loss = self._train_epoch(
#                     train_loader,
#                     loss_fn,
#                     optimizer,
#                     kl_weight,
#                     lambda_ewc if task_idx > 0 else 0.0,
#                     mc_samples=mc_samples_train
#                 )

#                 val_metric = self._validate(
#                     val_loader,
#                     task,
#                     mc_samples=mc_samples_val
#                 )

#                 self.training_history[task].append({
#                     'epoch': epoch,
#                     'train_loss': train_loss,
#                     'val_metric': val_metric
#                 })

#                 print(f"Epoch {epoch+1}/{num_epochs_per_task} | "
#                       f"Loss: {train_loss:.4f} | Val: {val_metric:.4f}")

#                 if val_metric > best_val_metric:
#                     best_val_metric = val_metric
#                     patience_counter = 0
#                     self._save_checkpoint(task, epoch, val_metric)
#                 else:
#                     patience_counter += 1

#                 scheduler.step(val_metric)

#                 if patience_counter >= patience:
#                     print("Early stopping triggered")
#                     break


#             if task_idx < len(self.tasks) - 1:
#                 print("\nComputing Fisher Information...")
#                 self.ewc_regularizer.compute_fisher_information(
#                     train_loader,
#                     loss_fn,
#                     self.device
#                 )

#     # =====================================================
#     # TRAIN EPOCH
#     # =====================================================
#     def _train_epoch(
#         self,
#         dataloader,
#         loss_fn,
#         optimizer,
#         kl_weight,
#         ewc_weight,
#         mc_samples=1
#     ):
#         self.model.train()

#         total_loss = 0.0

#         num_batches = len(dataloader)

#         for images, targets in tqdm(dataloader, desc="Training", leave=False):

#             images = images.to(self.device)
#             targets = targets.to(self.device)

#             optimizer.zero_grad()

#             # Get predictions and uncertainty
#             output = self.model(
#                 images,
#                 sample=True,
#                 num_mc_samples=mc_samples,
#                 return_uncertainty=True
#             )
#             predictions = output['predictions']

#             # KL divergence
#             kl_div = self.model.get_kl_divergence() / num_batches

#             loss = loss_fn(predictions, targets, kl_div, kl_weight)

#             # EWC regularization
#             if ewc_weight > 0:
#                 loss = loss + ewc_weight * self.ewc_regularizer.get_ewc_loss()

#             loss.backward()

#             torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)

#             optimizer.step()

#             total_loss += loss.item()

#         return total_loss / num_batches

#     # =====================================================
#     # VALIDATION
#     # =====================================================
#     def _validate(self, dataloader, task, mc_samples=1):
#         self.model.eval()

#         total_metric = 0.0
#         num_batches = 0

#         with torch.no_grad():
#             for images, targets in dataloader:

#                 images = images.to(self.device)
#                 targets = targets.to(self.device)

#                 # Get predictions and uncertainty
#                 output = self.model(
#                     images,
#                     sample=True,
#                     num_mc_samples=mc_samples,
#                     return_uncertainty=True
#                 )
#                 predictions = output['predictions']
#                 uncertainty = output['uncertainty']

#                 metric = self._compute_task_metric(predictions, targets, task)

#                 total_metric += metric
#                 num_batches += 1

#         return total_metric / max(num_batches, 1)

#     # =====================================================
#     # METRICS
#     # =====================================================
#     def _compute_task_metric(self, predictions, targets, task):

#         if task in ['detection', 'oriented']:
#             iou = self._compute_iou(predictions[:, :4], targets[:, :4])
#             return iou.mean().item()

#         elif task == 'segmentation':
#             probs = torch.sigmoid(predictions)

#             intersection = (probs * targets).sum(dim=1)
#             union = probs.sum(dim=1) + targets.sum(dim=1)

#             dice = (2 * intersection + 1e-8) / (union + 1e-8)
#             return dice.mean().item()

#         elif task == 'pose':
#             dist = torch.norm(predictions - targets, dim=1).mean()
#             return -dist.item()

#         elif task == 'classification':
#             preds = predictions.argmax(dim=1)
#             targets = targets.argmax(dim=1) if targets.dim() == 2 else targets
#             return (preds == targets).float().mean().item()

#         return 0.0

#     # =====================================================
#     # IOU
#     # =====================================================
#     def _compute_iou(self, pred, target):

#         x1 = torch.max(pred[:, 0], target[:, 0])
#         y1 = torch.max(pred[:, 1], target[:, 1])
#         x2 = torch.min(pred[:, 2], target[:, 2])
#         y2 = torch.min(pred[:, 3], target[:, 3])

#         inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

#         area_p = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
#         area_t = (target[:, 2] - target[:, 0]) * (target[:, 3] - target[:, 1])

#         union = area_p + area_t - inter

#         return inter / (union + 1e-8)

#     # =====================================================
#     # CHECKPOINTING
#     # =====================================================
#     def _save_checkpoint(self, task, epoch, metric):

#         path = self.checkpoint_dir / f"b_yolo_{task}_best.pt"

#         torch.save({
#             'epoch': epoch,
#             'model_state_dict': self.model.state_dict(),
#             'metric': metric,
#             'fisher': self.ewc_regularizer.fisher_information,
#             'prev_weights': self.ewc_regularizer.previous_weights
#         }, path)

#         self.best_metrics[task]['score'] = metric
#         self.best_metrics[task]['model_path'] = str(path)

#         print(f" Saved best {task} model: {metric:.4f}")


# import torch
# import torch.optim as optim
# from pathlib import Path
# from collections import defaultdict
# from tqdm import tqdm

class B_YOLOTrainer:
    def __init__(
        self,
        model,
        tasks,
        device=None,
        checkpoint_dir='./checkpoints'
    ):
        self.model = model
        self.tasks = tasks
        # self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        self.ewc_regularizer = EWCRegularizer(model)

        self.best_metrics = defaultdict(lambda: {'score': 0.0, 'model_path': None})
        self.training_history = defaultdict(list)

    # =====================================================
    # MAIN TRAINING LOOP
    # =====================================================
    def train_sequential_tasks(
        self,
        task_dataloaders,
        num_epochs_per_task=50,
        learning_rate=1e-3,
        kl_weight=0.01,
        lambda_ewc=0.4,
        mc_samples_train=1,
        mc_samples_val=3
    ):
        self.model.to(self.device)

        for task_idx, task in enumerate(self.tasks):

            print(f"\n{'='*60}")
            print(f"Task {task_idx+1}/{len(self.tasks)}: {task.upper()}")
            print(f"{'='*60}")

            self.model.set_task(task)

            loss_fn = TaskSpecificLoss(task, self.model.num_classes)

            optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, self.model.parameters()),
                lr=learning_rate
            )

            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=5
            )

            train_loader = task_dataloaders[task]['train']
            val_loader = task_dataloaders[task]['val']

            best_val_metric = -float('inf')
            patience_counter = 0
            patience = 10

            for epoch in range(num_epochs_per_task):

                train_loss = self._train_epoch(
                    train_loader,
                    loss_fn,
                    optimizer,
                    kl_weight,
                    lambda_ewc if task_idx > 0 else 0.0,
                    mc_samples=mc_samples_train
                )

                val_metric = self._validate(
                    val_loader,
                    task,
                    mc_samples=mc_samples_val
                )

                self.training_history[task].append({
                    'epoch': epoch,
                    'train_loss': train_loss,
                    'val_metric': val_metric
                })

                print(f"Epoch {epoch+1}/{num_epochs_per_task} | "
                      f"Loss: {train_loss:.4f} | Val: {val_metric:.4f}")

                if val_metric > best_val_metric:
                    best_val_metric = val_metric
                    patience_counter = 0
                    self._save_checkpoint(task, epoch, val_metric)
                else:
                    patience_counter += 1

                scheduler.step(val_metric)

                if patience_counter >= patience:
                    print("Early stopping triggered")
                    break


            if task_idx < len(self.tasks) - 1:
                print("\nComputing Fisher Information...")
                self.ewc_regularizer.compute_fisher_information(
                    train_loader,
                    loss_fn,
                    self.device
                )

    # =====================================================
    # TRAIN EPOCH
    # =====================================================
    def _train_epoch(
        self,
        dataloader,
        loss_fn,
        optimizer,
        kl_weight,
        ewc_weight,
        mc_samples=1
    ):
        self.model.train()

        total_loss = 0.0
        num_batches = len(dataloader)

        for images, targets in tqdm(dataloader, desc="Training", leave=False):

            # =====================================================
            # CHANGE 1: Handle Tuple to Tensor Conversion Safely
            # =====================================================
            if isinstance(images, tuple) or isinstance(images, list):
                images = torch.stack(images).to(self.device)
            else:
                images = images.to(self.device)
                
            if isinstance(targets, tuple) or isinstance(targets, list):
                # If targets are COCO dicts, move inner tensors to device
                # targets = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
                targets = [[{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in ann.items()} for ann in t] for t in targets]
            else:
                targets = targets.to(self.device)
            # =====================================================

            optimizer.zero_grad()

            # Get predictions and uncertainty
            output = self.model(
                images,
                sample=True,
                num_mc_samples=mc_samples,
                return_uncertainty=True
            )
            predictions = output['predictions']

            # KL divergence
            kl_div = self.model.get_kl_divergence() / num_batches

            loss = loss_fn(predictions, targets, kl_div, kl_weight)

            # EWC regularization
            if ewc_weight > 0:
                loss = loss + ewc_weight * self.ewc_regularizer.get_ewc_loss()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)

            optimizer.step()

            total_loss += loss.item()

        return total_loss / num_batches

    # =====================================================
    # VALIDATION
    # =====================================================
    def _validate(self, dataloader, task, mc_samples=1):
        self.model.eval()

        total_metric = 0.0
        num_batches = 0

        with torch.no_grad():
            for images, targets in dataloader:

                # =====================================================
                # CHANGE 2: Handle Tuple to Tensor Conversion Safely
                # =====================================================
                if isinstance(images, tuple) or isinstance(images, list):
                    images = torch.stack(images).to(self.device)
                else:
                    images = images.to(self.device)
                    
                if isinstance(targets, tuple) or isinstance(targets, list):
                    # targets = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
                    targets = [[{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in ann.items()} for ann in t] for t in targets]

                else:
                    targets = targets.to(self.device)
                # =====================================================

                # Get predictions and uncertainty
                output = self.model(
                    images,
                    sample=True,
                    num_mc_samples=mc_samples,
                    return_uncertainty=True
                )
                predictions = output['predictions']
                uncertainty = output['uncertainty']

                metric = self._compute_task_metric(predictions, targets, task)

                total_metric += metric
                num_batches += 1

        return total_metric / max(num_batches, 1)

    # =====================================================
    # METRICS
    # =====================================================
    # def _compute_task_metric(self, predictions, targets, task):

    #     if task in ['detection', 'oriented']:
    #         # NOTE: If targets is a list of COCO dicts, you will need to parse 
    #         # the bounding boxes out of the dicts here before passing to _compute_iou!
    #         iou = self._compute_iou(predictions[:, :4], targets[:, :4])
    #         return iou.mean().item()

    #     elif task == 'segmentation':
    #         probs = torch.sigmoid(predictions)

    #         intersection = (probs * targets).sum(dim=1)
    #         union = probs.sum(dim=1) + targets.sum(dim=1)

    #         dice = (2 * intersection + 1e-8) / (union + 1e-8)
    #         return dice.mean().item()

    #     elif task == 'pose':
    #         dist = torch.norm(predictions - targets, dim=1).mean()
    #         return -dist.item()

    #     elif task == 'classification':
    #         preds = predictions.argmax(dim=1)
    #         targets = targets.argmax(dim=1) if targets.dim() == 2 else targets
    #         return (preds == targets).float().mean().item()

    #     return 0.0
    # =====================================================
    # New Metrics function
    # =====================================================
    def _compute_task_metric(self, predictions, targets, task):

        if task in ['detection', 'oriented']:
            batch_size = predictions.size(0)
            total_iou = 0.0
            objects_found = 0

            # Loop through batch image by image
            for i in range(batch_size):
                pred_for_image = predictions[i]
                
                # Un-flatten if the model squashed the output (just like in the loss function)
                if pred_for_image.dim() == 1:
                    # predictions have 4 bbox coords + num_classes
                    num_features = 4 + self.model.num_classes 
                    pred_for_image = pred_for_image.view(-1, num_features)
                
                pred_boxes = pred_for_image[:, :4]
                img_targets = targets[i]

                if len(img_targets) == 0:
                    continue

                # 1. Parse COCO dictionaries into a Ground Truth Tensor
                gt_boxes = []
                for ann in img_targets:
                    bbox = ann['bbox']
                    if not isinstance(bbox, torch.Tensor):
                        bbox = torch.tensor(bbox, device=predictions.device, dtype=torch.float32)
                    gt_boxes.append(bbox)
                
                gt_boxes = torch.stack(gt_boxes) # Shape: [num_gt_objects, 4]
                objects_found += gt_boxes.size(0)

                # 2. Naive Assignment (Match predictions to ground truth)
                # We use the same L1 distance matching here so our metrics reflect our loss
                distances = torch.cdist(pred_boxes, gt_boxes, p=1)
                best_pred_indices = torch.argmin(distances, dim=0)
                matched_pred_boxes = pred_boxes[best_pred_indices]

                # 3. Calculate IoU only on the matched pairs
                iou = self._compute_iou(matched_pred_boxes, gt_boxes)
                total_iou += iou.sum().item()

            # Return the average IoU across all objects in the batch
            if objects_found > 0:
                return total_iou / objects_found
            return 0.0

        elif task == 'segmentation':
            probs = torch.sigmoid(predictions)
            intersection = (probs * targets).sum(dim=1)
            union = probs.sum(dim=1) + targets.sum(dim=1)
            dice = (2 * intersection + 1e-8) / (union + 1e-8)
            return dice.mean().item()

        elif task == 'pose':
            # dist = torch.norm(predictions - targets, dim=1).mean()
            # return -dist.item()
            batch_size = predictions.size(0)
            total_dist = 0.0
            objects_found = 0

            for i in range(batch_size):
                pred_for_image = predictions[i]
                if pred_for_image.dim() == 1:
                    pred_for_image = pred_for_image.view(-1, 34)

                img_targets = targets[i]
                if len(img_targets) == 0:
                    continue

                # 1. Extract the ground truth poses from the COCO dictionaries
                gt_poses = []
                for ann in img_targets:
                    if 'keypoints' not in ann:
                        continue
                    
                    kpts = ann['keypoints']
                    if not isinstance(kpts, torch.Tensor):
                        kpts = torch.tensor(kpts, device=predictions.device, dtype=torch.float32)
                    
                    # Strip out visibility flags (keep only x and y)
                    xy_indices = [j for j in range(51) if j % 3 != 2]
                    kpts_xy = kpts[xy_indices]
                    gt_poses.append(kpts_xy)

                if len(gt_poses) == 0:
                    continue

                gt_poses = torch.stack(gt_poses)
                objects_found += gt_poses.size(0)

                # 2. Match predictions to ground truth
                distances = torch.cdist(pred_for_image, gt_poses, p=2) 
                best_pred_indices = torch.argmin(distances, dim=0)
                matched_pred_poses = pred_for_image[best_pred_indices]

                # 3. Calculate the L2 Distance (Norm) on matched pairs
                # Calculate the distance per object, then sum it up for the image
                dist = torch.norm(matched_pred_poses - gt_poses, dim=1).sum()
                total_dist += dist.item()

            # Return the average distance across all valid objects
            if objects_found > 0:
                # We return negative distance because trainers usually assume "higher metric = better"
                # (A distance of 0 is perfect, so -0.0 is better than -50.0)
                return -(total_dist / objects_found)
            else:
                return 0.0

        elif task == 'classification':
            preds = predictions.argmax(dim=1)
            targets = targets.argmax(dim=1) if targets.dim() == 2 else targets
            return (preds == targets).float().mean().item()

        return 0.0


    # =====================================================
    # IOU
    # =====================================================
    def _compute_iou(self, pred, target):

        x1 = torch.max(pred[:, 0], target[:, 0])
        y1 = torch.max(pred[:, 1], target[:, 1])
        x2 = torch.min(pred[:, 2], target[:, 2])
        y2 = torch.min(pred[:, 3], target[:, 3])

        inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

        area_p = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
        area_t = (target[:, 2] - target[:, 0]) * (target[:, 3] - target[:, 1])

        union = area_p + area_t - inter

        return inter / (union + 1e-8)

    # =====================================================
    # CHECKPOINTING
    # =====================================================
    def _save_checkpoint(self, task, epoch, metric):

        path = self.checkpoint_dir / f"b_yolo_{task}_best.pt"

        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'metric': metric,
            'fisher': self.ewc_regularizer.fisher_information,
            'prev_weights': self.ewc_regularizer.previous_weights
        }, path)

        self.best_metrics[task]['score'] = metric
        self.best_metrics[task]['model_path'] = str(path)

        print(f" Saved best {task} model: {metric:.4f}")

In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader, TensorDataset


# class DummyBackbone(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.conv = nn.Conv2d(3, 512, 1)
#     def forward(self, x):
#         return self.conv(x)

# # 2. Initialize Model
# # backbone = DummyBackbone()
# backbone = model_cls
# model = B_YOLO(backbone, num_classes=10)


# def create_dummy_loader(num_samples=10, input_shape=(3, 224, 224), num_classes=10):
#     x = torch.randn(num_samples, *input_shape)
#     y = torch.randint(0, num_classes, (num_samples,))
#     dataset = TensorDataset(x, y)
#     return DataLoader(dataset, batch_size=2)

# task_dataloaders = {
#     'classification': {
#         'train': create_dummy_loader(),
#         'val': create_dummy_loader()
#     }
# }


# trainer = B_YOLOTrainer(model, tasks=['classification'])


# try:
#     print("Starting smoke test...")
#     trainer.train_sequential_tasks(
#         task_dataloaders,
#         num_epochs_per_task=1,
#         learning_rate=1e-4,
#         mc_samples_train=1,  # Single sample during training
#         mc_samples_val=2     # Multiple samples during validation for uncertainty
#     )
#     print("\n Success! The pipeline runs correctly.")

#     # Test uncertainty output
#     print("\nTesting uncertainty output...")
#     model.eval()
#     sample_input = torch.randn(2, 3, 224, 224)

#     # Test with single sample (no uncertainty)
#     output_single = model(sample_input, num_mc_samples=1, return_uncertainty=True)
#     print(f" Single forward pass:")
#     print(f"  - Predictions shape: {output_single['predictions'].shape}")
#     print(f"  - Uncertainty shape: {output_single['uncertainty'].shape}")
#     print(f"  - Uncertainty (should be mostly zeros): {output_single['uncertainty'].mean().item():.6f}")

#     # Test with multiple samples (with uncertainty)
#     output_multi = model(sample_input, num_mc_samples=5, return_uncertainty=True)
#     print(f"\n Multiple MC forward passes (5 samples):")
#     print(f"  - Predictions shape: {output_multi['predictions'].shape}")
#     print(f"  - Uncertainty shape: {output_multi['uncertainty'].shape}")
#     print(f"  - Uncertainty (should be non-zero): {output_multi['uncertainty'].mean().item():.6f}")

# except Exception as e:
#     print(f"\nAn error occurred: {e}")
#     import traceback
#     traceback.print_exc()


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import CocoDetection
from torchvision.transforms import v2
from ultralytics import YOLO

# Assume B_YOLO and B_YOLOTrainer are imported from your custom module
# from your_module import B_YOLO, B_YOLOTrainer

# 1. Real YOLOv11s Backbone
# class YOLO11sBackbone(nn.Module):
#     def __init__(self, pretrained=True):
#         super().__init__()
#         # Load the official ultralytics model
#         model_name = "yolo11s.pt" if pretrained else "yolo11s.yaml"
#         self.yolo = YOLO(model_name)
        
#         # Extract the backbone from the Ultralytics PyTorch model.
#         # In YOLO architectures, the backbone typically consists of the first 10 layers 
#         # (ending with the SPPF block).
#         self.backbone = self.yolo.model.model[:10]
        
#     def forward(self, x):
#         # Pass the input through the sequential backbone layers
#         for layer in self.backbone:
#             x = layer(x)
#         return x

class YOLO11sBackbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        # 1. Load the official ultralytics model as a LOCAL variable (no "self.")
        model_name = "yolo11s.pt" if pretrained else "yolo11s.yaml"
        yolo = YOLO(model_name)
        
        # 2. Extract the PyTorch backbone layers and store THEM as the module attribute
        # yolo.model is the actual PyTorch nn.Module. 
        # yolo.model.model is the Sequential container of layers.
        self.backbone = yolo.model.model[:10]
        
    def forward(self, x):
        # Pass the input through the sequential backbone layers
        for layer in self.backbone:
            x = layer(x)
        return x

# 2. Real Object Detection Dataset Loader (COCO Format)
def collate_fn(batch):
    # Standard collate function for object detection to handle variable 
    # numbers of bounding boxes per image.
    return tuple(zip(*batch))

def create_coco_loader(image_dir, annotation_json, batch_size=2, input_size=(640, 640)):
    # Modern torchvision v2 transforms handle both images and bounding boxes
    transforms = v2.Compose([
        v2.ToImage(),
        v2.Resize(input_size),
        v2.ToDtype(torch.float32, scale=True),
    ])
    
    # Load the dataset using the standard COCO class
    dataset = CocoDetection(
        root=image_dir,
        annFile=annotation_json,
        transforms=transforms
    )
    
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True if 'train' in annotation_json.lower() else False,
        collate_fn=collate_fn,
        num_workers=4,
        pin_memory=True
    )

# 3. Initialize Model
backbone = YOLO11sBackbone(pretrained=True)

# Note: Adjust num_classes based on your specific dataset.
# The standard COCO dataset uses 80 classes.
model = B_YOLO(backbone, num_classes=80) 

# 4. Set up Real DataLoaders
task_dataloaders = {
    'detection': {
        'train': create_coco_loader(
            image_dir ='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/train2017', 
            annotation_json = '/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/1_Object_Detection/train.json', 
            batch_size=4
        ),
        'val': create_coco_loader(
            image_dir = '/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/train2017', 
            annotation_json = '/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/1_Object_Detection/test.json', 
            batch_size=4
        )
    }
}

trainer = B_YOLOTrainer(model, tasks=['detection'])

# 5. Run the Pipeline
try:
    print("Starting training pipeline...")
    trainer.train_sequential_tasks(
        task_dataloaders,
        num_epochs_per_task=5,
        learning_rate=1e-4,
        mc_samples_train=1,  # Single sample during training
        mc_samples_val=2     # Multiple samples during validation for uncertainty
    )
    print("\n Success! The pipeline runs correctly.")

except Exception as e:
    print(f"\nAn error occurred: {e}")
    import traceback
    traceback.print_exc()

loading annotations into memory...
Done (t=0.44s)
creating index...
index created!
loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Starting training pipeline...

Task 1/1: DETECTION


KeyboardInterrupt: 

In [49]:

# Test uncertainty output
print("\nTesting uncertainty output...")
model.eval()
val_loader = task_dataloaders['detection']['val']
images, targets = next(iter(val_loader))

# 2. Safely convert the tuple of images into a tensor and move to GPU
if isinstance(images, tuple) or isinstance(images, list):
    _input = torch.stack(images).to(trainer.device)
else:
    _input = images.to(trainer.device)

# Test with single sample (no uncertainty)
output_single = model(_input, num_mc_samples=1, return_uncertainty=True)
print(f" Single forward pass:")
print(f"  - Predictions shape: {output_single['predictions'].shape}")
print(f"  - Uncertainty shape: {output_single['uncertainty'].shape}")
print(f"  - Uncertainty (should be mostly zeros): {output_single['uncertainty'].mean().item():.6f}")

# Test with multiple samples (with uncertainty)
output_multi = model(_input, num_mc_samples=5, return_uncertainty=True)
print(f"\n Multiple MC forward passes (5 samples):")
print(f"  - Predictions shape: {output_multi['predictions'].shape}")
print(f"  - Uncertainty shape: {output_multi['uncertainty'].shape}")
print(f"  - Uncertainty (should be non-zero): {output_multi['uncertainty'].mean().item():.6f}")


Testing uncertainty output...


 Single forward pass:
  - Predictions shape: torch.Size([4, 84])
  - Uncertainty shape: torch.Size([4, 84])
  - Uncertainty (should be mostly zeros): 0.000000

 Multiple MC forward passes (5 samples):
  - Predictions shape: torch.Size([4, 84])
  - Uncertainty shape: torch.Size([4, 84])
  - Uncertainty (should be non-zero): 1.028251


In [ ]:

# Test uncertainty output
print("\nTesting uncertainty output...")
model.eval()
val_loader = task_dataloaders['detection']['val']
images, targets = next(iter(val_loader))

# 2. Safely convert the tuple of images into a tensor and move to GPU
if isinstance(images, tuple) or isinstance(images, list):
    _input = torch.stack(images).to(trainer.device)
else:
    _input = images.to(trainer.device)

# Test with single sample (no uncertainty)
output_single = model(_input, num_mc_samples=1, return_uncertainty=True)
print(f" Single forward pass:")
print(f"  - Predictions shape: {output_single['predictions'].shape}")
print(f"  - Uncertainty shape: {output_single['uncertainty'].shape}")
print(f"  - Uncertainty (should be mostly zeros): {output_single['uncertainty'].mean().item():.6f}")

# Test with multiple samples (with uncertainty)
output_multi = model(_input, num_mc_samples=5, return_uncertainty=True)
print(f"\n Multiple MC forward passes (5 samples):")
print(f"  - Predictions shape: {output_multi['predictions'].shape}")
print(f"  - Uncertainty shape: {output_multi['uncertainty'].shape}")
print(f"  - Uncertainty (should be non-zero): {output_multi['uncertainty'].mean().item():.6f}")


import torch
import torch.nn as nn
import numpy as np

print("="*70)
print("COMPREHENSIVE REAL DATA INPUT & OUTPUT DEMONSTRATION (DETECTION)")
print("="*70)

# 1. Fetch REAL data from your validation dataloader
print("Fetching a batch of real images from the detection dataloader...")
val_loader = task_dataloaders['detection']['val']
images, targets = next(iter(val_loader))

# Determine the device the model is currently on
device = next(model.parameters()).device

# Safely handle tuple/list of images (standard for object detection) and move to GPU
if isinstance(images, (tuple, list)):
    real_input = torch.stack(images).to(device)
else:
    real_input = images.to(device)

print(f"Real Input shape: {real_input.shape} (batch_size, channels, height, width)")

# Set model to evaluation mode and configure for detection
model.eval()
model.set_task('detection')

# =====================================================================
# Test 1: SINGLE FORWARD PASS (No Uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 1: SINGLE FORWARD PASS (num_mc_samples=1)")
print("-"*70)

with torch.no_grad():
    output_single = model(real_input, num_mc_samples=1, return_uncertainty=False)

print(f"Output predictions shape: {output_single['predictions'].shape}")
print(f"  - First 4 values: bounding boxes")
print(f"  - Remaining values: class logits")

print(f"\nDetection Output for Sample 1:")
# Note: Using .cpu() before .numpy() ensures it works even if data is on the GPU
if output_single['predictions'].dim() == 2:
    # Shape: [batch_size, features]
    bbox = output_single['predictions'][0, :4]
    classes = output_single['predictions'][0, 4:]
else:
    # Shape: [batch_size, num_anchors, features]
    bbox = output_single['predictions'][0, 0, :4]
    classes = output_single['predictions'][0, 0, 4:]

print(f"  Bounding Box: {bbox.cpu().detach().numpy()}")
print(f"  Class Logits (first 5): {classes[:5].cpu().detach().numpy()}...")


# =====================================================================
# Test 2: MULTIPLE FORWARD PASSES (With Uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 2: MULTIPLE MC PASSES (num_mc_samples=10) WITH UNCERTAINTY")
print("-"*70)

with torch.no_grad():
    output_multi = model(real_input, num_mc_samples=10, return_uncertainty=True)

print(f"Mean Predictions shape: {output_multi['predictions'].shape}")
print(f"Uncertainty shape: {output_multi['uncertainty'].shape}")

print(f"\nUncertainty for Sample 1:")
if output_multi['uncertainty'].dim() == 2:
    unc_bbox = output_multi['uncertainty'][0, :4]
    unc_classes = output_multi['uncertainty'][0, 4:]
else:
    unc_bbox = output_multi['uncertainty'][0, 0, :4]
    unc_classes = output_multi['uncertainty'][0, 0, 4:]

print(f"  BBox Uncertainty: {unc_bbox.cpu().detach().numpy()}")
print(f"  Class Uncertainty (first 5): {unc_classes[:5].cpu().detach().numpy()}...")
print(f"  Mean overall uncertainty for Image 1: {output_multi['uncertainty'][0].mean().item():.6f}")


# =====================================================================
# Test 3: DETAILED MC INFERENCE (predict_with_uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 3: DETAILED MC INFERENCE (T=20)")
print("-"*70)

with torch.no_grad():
    # Take just the first real image from the batch
    sample = real_input[:1]  
    
    # Run the custom B_YOLO method
    mc_output = model.predict_with_uncertainty(sample, T=20)

print(f"Using T=20 MC forward passes on 1 real sample:")
# print(f"Uncertainty shape: {mc_output['uncertainty'].shape}")
print(f"mc_output : {mc_output}")

if 'confidence' in mc_output:
    print(f"Confidence shape: {mc_output['confidence'].shape}")

print("\n" + "="*70)
print(" All detection tests with REAL DATA completed successfully!")
print("="*70)

COMPREHENSIVE REAL DATA INPUT & OUTPUT DEMONSTRATION (DETECTION)
Fetching a batch of real images from the detection dataloader...


Real Input shape: torch.Size([4, 3, 640, 640]) (batch_size, channels, height, width)

----------------------------------------------------------------------
TEST 1: SINGLE FORWARD PASS (num_mc_samples=1)
----------------------------------------------------------------------
Output predictions shape: torch.Size([4, 84])
  - First 4 values: bounding boxes
  - Remaining values: class logits

Detection Output for Sample 1:
  Bounding Box: [     294.82       266.7      87.884      97.588]
  Class Logits (first 5): [    -39.146      4.0547   -0.079507      1.6402    -0.18062]...

----------------------------------------------------------------------
TEST 2: MULTIPLE MC PASSES (num_mc_samples=10) WITH UNCERTAINTY
----------------------------------------------------------------------
Mean Predictions shape: torch.Size([4, 84])
Uncertainty shape: torch.Size([4, 84])

Uncertainty for Sample 1:
  BBox Uncertainty: [     63.924       63.21      19.769      27.385]
  Class Uncertainty (first 5): [ 

# Classification Task 

In [6]:
import json

with open('/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/5_Classification/train.json', 'r') as f:
    data = json.load(f)

# Extract all category_ids and find the maximum
max_id = max(ann['category_id'] for ann in data['annotations'])

print(f"Maximum category_id found: {max_id}")
print(f"So Max num_classes={max_id + 1}")

Maximum category_id found: 101
So Max num_classes=102


In [7]:
import torch
import json
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader,Dataset
from torchvision.datasets import CocoDetection
from torchvision.transforms import v2
from ultralytics import YOLO

# Assume B_YOLO and B_YOLOTrainer are imported from your custom module
class YOLO11sBackbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        # 1. Load the official ultralytics model as a LOCAL variable (no "self.")
        model_name = "yolo11s.pt" if pretrained else "yolo11s.yaml"
        yolo = YOLO(model_name)
        
        # 2. Extract the PyTorch backbone layers and store THEM as the module attribute
        # yolo.model is the actual PyTorch nn.Module. 
        # yolo.model.model is the Sequential container of layers.
        self.backbone = yolo.model.model[:10]
        
    def forward(self, x):
        # Pass the input through the sequential backbone layers
        for layer in self.backbone:
            x = layer(x)
        return x

class JSONClassificationDataset(Dataset):
    def __init__(self, image_dir, json_path, transforms=None):
        self.image_dir = image_dir
        self.transforms = transforms
        
        # Load the JSON file
        with open(json_path, 'r') as f:
            data = json.load(f)
            
        self.samples = []
        
        # Scenario A: COCO-style dictionary {"images": [...], "annotations": [...]}
        if isinstance(data, dict) and 'annotations' in data:
            img_dict = {img['id']: img['file_name'] for img in data['images']}
            for ann in data['annotations']:
                self.samples.append({
                    'file_name': img_dict[ann['image_id']],
                    'label': ann['category_id']
                })
                
        # Scenario B: Simple list of dictionaries [{"file_name": "img1.jpg", "label": 3}]
        elif isinstance(data, list):
            self.samples = data
            
        else:
            raise ValueError("Unrecognized JSON format.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Gracefully handle whatever keys your JSON uses
        img_name = sample.get('file_name', sample.get('image', sample.get('filename')))
        label = sample.get('label', sample.get('category_id', sample.get('class_id')))
        
        # Load the image
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        
        # Apply transforms
        if self.transforms:
            image = self.transforms(image)
            
        # Return the tensor and the integer label!
        return image, torch.tensor(label, dtype=torch.long)

def create_json_class_loader(image_dir, annotation_json, batch_size=4, input_size=(224, 224), is_train=True):
    transforms = v2.Compose([
        v2.ToImage(),
        v2.Resize(input_size),
        v2.ToDtype(torch.float32, scale=True),
    ])
    
    dataset = JSONClassificationDataset(
        image_dir=image_dir,
        json_path=annotation_json,
        transforms=transforms
    )
    
    # Notice we do NOT use the custom collate_fn here!
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=is_train,
        num_workers=4,
        pin_memory=True
    )

# 3. Initialize Model
backbone = YOLO11sBackbone(pretrained=True)

# Note: Adjust num_classes based on your specific dataset.
# The standard COCO dataset uses 80 classes.
model = B_YOLO(backbone, num_classes=102) 

# 4. Set up Real DataLoaders
task_dataloaders = {
    'classification': {
        'train': create_json_class_loader(
            image_dir='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/5_Classification/images', 
            annotation_json='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/5_Classification/train.json', 
            batch_size=4,
            is_train=True
        ),
        'val': create_json_class_loader(
            image_dir='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/5_Classification/images', 
            annotation_json='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/5_Classification/test.json', 
            batch_size=4,
            is_train=False
        )
    }
}

trainer = B_YOLOTrainer(model, tasks=['classification'])

# 5. Run the Pipeline
try:
    print("Starting training pipeline...")
    trainer.train_sequential_tasks(
        task_dataloaders,
        num_epochs_per_task=5,
        learning_rate=1e-4,
        mc_samples_train=1,  # Single sample during training
        mc_samples_val=2     # Multiple samples during validation for uncertainty
    )
    print("\n Success! The pipeline runs correctly.")

except Exception as e:
    print(f"\nAn error occurred: {e}")
    import traceback
    traceback.print_exc()

Starting training pipeline...

Task 1/1: CLASSIFICATION


Epoch 1/5 | Loss: 5.4174 | Val: 0.0160
 Saved best classification model: 0.0160


Epoch 2/5 | Loss: 5.3786 | Val: 0.0337
 Saved best classification model: 0.0337


Epoch 3/5 | Loss: 5.3206 | Val: 0.0483
 Saved best classification model: 0.0483


Epoch 4/5 | Loss: 5.2422 | Val: 0.0700
 Saved best classification model: 0.0700


Epoch 5/5 | Loss: 5.1551 | Val: 0.0837
 Saved best classification model: 0.0837

 Success! The pipeline runs correctly.


In [9]:
# Test uncertainty output
print("\nTesting uncertainty output...")
model.eval()
val_loader = task_dataloaders['classification']['val']
images, targets = next(iter(val_loader))

# 2. Safely convert the tuple of images into a tensor and move to GPU
if isinstance(images, tuple) or isinstance(images, list):
    _input = torch.stack(images).to(trainer.device)
else:
    _input = images.to(trainer.device)

# Test with single sample (no uncertainty)
output_single = model(_input, num_mc_samples=1, return_uncertainty=True)
print(f" Single forward pass:")
print(f"  - Predictions shape: {output_single['predictions'].shape}")
print(f"  - Uncertainty shape: {output_single['uncertainty'].shape}")
print(f"  - Uncertainty (should be mostly zeros): {output_single['uncertainty'].mean().item():.6f}")

# Test with multiple samples (with uncertainty)
output_multi = model(_input, num_mc_samples=5, return_uncertainty=True)
print(f"\n Multiple MC forward passes (5 samples):")
print(f"  - Predictions shape: {output_multi['predictions'].shape}")
print(f"  - Uncertainty shape: {output_multi['uncertainty'].shape}")
print(f"  - Uncertainty (should be non-zero): {output_multi['uncertainty'].mean().item():.6f}")




Testing uncertainty output...
 Single forward pass:
  - Predictions shape: torch.Size([4, 102])
  - Uncertainty shape: torch.Size([4, 102])
  - Uncertainty (should be mostly zeros): 0.000000

 Multiple MC forward passes (5 samples):
  - Predictions shape: torch.Size([4, 102])
  - Uncertainty shape: torch.Size([4, 102])
  - Uncertainty (should be non-zero): 0.331650


In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("="*70)
print("COMPREHENSIVE REAL DATA INPUT & OUTPUT DEMONSTRATION (CLASSIFICATION)")
print("="*70)

# 1. Fetch REAL data from your validation dataloader
print("Fetching a batch of real images from the detection dataloader...")
val_loader = task_dataloaders['classification']['val']
images, targets = next(iter(val_loader))

# Determine the device the model is currently on
device = next(model.parameters()).device

# Safely handle tuple/list of images (standard for object detection) and move to GPU
if isinstance(images, (tuple, list)):
    real_input = torch.stack(images).to(device)
else:
    real_input = images.to(device)

print(f"Real Input shape: {real_input.shape} (batch_size, channels, height, width)")

# Set model to evaluation mode and configure for detection
model.eval()
model.set_task('detection')

# =====================================================================
# Test 1: SINGLE FORWARD PASS (No Uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 1: SINGLE FORWARD PASS (num_mc_samples=1)")
print("-"*70)

with torch.no_grad():
    output_single = model(real_input, num_mc_samples=1, return_uncertainty=False)

print(f"Output predictions shape: {output_single['predictions'].shape}")
print(f"  - First 4 values: bounding boxes")
print(f"  - Remaining values: class logits")

print(f"\nDetection Output for Sample 1:")
# Note: Using .cpu() before .numpy() ensures it works even if data is on the GPU
if output_single['predictions'].dim() == 2:
    # Shape: [batch_size, features]
    bbox = output_single['predictions'][0, :4]
    classes = output_single['predictions'][0, 4:]
else:
    # Shape: [batch_size, num_anchors, features]
    bbox = output_single['predictions'][0, 0, :4]
    classes = output_single['predictions'][0, 0, 4:]

print(f"  Bounding Box: {bbox.cpu().detach().numpy()}")
print(f"  Class Logits (first 5): {classes[:5].cpu().detach().numpy()}...")


# =====================================================================
# Test 2: MULTIPLE FORWARD PASSES (With Uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 2: MULTIPLE MC PASSES (num_mc_samples=10) WITH UNCERTAINTY")
print("-"*70)

with torch.no_grad():
    output_multi = model(real_input, num_mc_samples=10, return_uncertainty=True)

print(f"Mean Predictions shape: {output_multi['predictions'].shape}")
print(f"Uncertainty shape: {output_multi['uncertainty'].shape}")

print(f"\nUncertainty for Sample 1:")
if output_multi['uncertainty'].dim() == 2:
    unc_bbox = output_multi['uncertainty'][0, :4]
    unc_classes = output_multi['uncertainty'][0, 4:]
else:
    unc_bbox = output_multi['uncertainty'][0, 0, :4]
    unc_classes = output_multi['uncertainty'][0, 0, 4:]

print(f"  BBox Uncertainty: {unc_bbox.cpu().detach().numpy()}")
print(f"  Class Uncertainty (first 5): {unc_classes[:5].cpu().detach().numpy()}...")
print(f"  Mean overall uncertainty for Image 1: {output_multi['uncertainty'][0].mean().item():.6f}")


# =====================================================================
# Test 3: DETAILED MC INFERENCE (predict_with_uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 3: DETAILED MC INFERENCE (T=20)")
print("-"*70)

with torch.no_grad():
    # Take just the first real image from the batch
    sample = real_input[:1]  
    
    # Run the custom B_YOLO method
    mc_output = model.predict_with_uncertainty(sample, T=20)

print(f"Using T=20 MC forward passes on 1 real sample:")
# print(f"Uncertainty shape: {mc_output['uncertainty'].shape}")
print(f"mc_output : {mc_output}")

if 'confidence' in mc_output:
    print(f"Confidence shape: {mc_output['confidence'].shape}")

print("\n" + "="*70)
print(" All calssification tests with REAL DATA completed successfully!")
print("="*70)

COMPREHENSIVE REAL DATA INPUT & OUTPUT DEMONSTRATION (DETECTION)
Fetching a batch of real images from the detection dataloader...
Real Input shape: torch.Size([4, 3, 224, 224]) (batch_size, channels, height, width)

----------------------------------------------------------------------
TEST 1: SINGLE FORWARD PASS (num_mc_samples=1)
----------------------------------------------------------------------
Output predictions shape: torch.Size([4, 106])
  - First 4 values: bounding boxes
  - Remaining values: class logits

Detection Output for Sample 1:
  Bounding Box: [  -0.078716     0.20502    0.014956     0.66791]
  Class Logits (first 5): [    0.34386    -0.47378    -0.50856     0.38382     0.81527]...

----------------------------------------------------------------------
TEST 2: MULTIPLE MC PASSES (num_mc_samples=10) WITH UNCERTAINTY
----------------------------------------------------------------------
Mean Predictions shape: torch.Size([4, 106])
Uncertainty shape: torch.Size([4, 106

## POSE/KEYPOINT DECTECTION

In [17]:
import torch
import json
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader,Dataset
from torchvision.datasets import CocoDetection
from torchvision.transforms import v2
from ultralytics import YOLO

# Assume B_YOLO and B_YOLOTrainer are imported from your custom module
class YOLO11sBackbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        # 1. Load the official ultralytics model as a LOCAL variable (no "self.")
        model_name = "yolo11s-pose.pt" if pretrained else "yolo11s-pose.yaml"
        yolo = YOLO(model_name)
        
        # 2. Extract the PyTorch backbone layers and store THEM as the module attribute
        # yolo.model is the actual PyTorch nn.Module. 
        # yolo.model.model is the Sequential container of layers.
        self.backbone = yolo.model.model[:10]
        
    def forward(self, x):
        # Pass the input through the sequential backbone layers
        for layer in self.backbone:
            x = layer(x)
        return x

def collate_fn(batch):
    # This keeps the complex JSON dictionaries intact!
    return tuple(zip(*batch))

def create_pose_loader(image_dir, annotation_json, batch_size=4, input_size=(640, 640)):
    transforms = v2.Compose([
        v2.ToImage(),
        v2.Resize(input_size),
        v2.ToDtype(torch.float32, scale=True),
    ])
    
    # CocoDetection automatically parses your complex JSON format
    dataset = CocoDetection(
        root=image_dir,
        annFile=annotation_json,
        transforms=transforms
    )
    
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True if 'train' in annotation_json.lower() else False,
        collate_fn=collate_fn, # <--- CRITICAL
        num_workers=4,
        pin_memory=True
    )

# 3. Initialize Model
backbone = YOLO11sBackbone(pretrained=True)

# Assuming 1 class (Person) for Pose Estimation
model = B_YOLO(backbone, num_classes=1) 

task_dataloaders = {
    'pose': {
        'train': create_pose_loader(
            image_dir='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/train2017', 
            annotation_json='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/3_Pose_Detection/train.json', 
            batch_size=4
        ),
        'val': create_pose_loader(
            image_dir='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/train2017', 
            annotation_json='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/3_Pose_Detection/test.json', 
            batch_size=4
        )
    }
}

# Ensure the trainer is set to 'pose'
trainer = B_YOLOTrainer(model, tasks=['pose'])

print("Starting POSE ESTIMATION training pipeline...")
trainer.train_sequential_tasks(
    task_dataloaders,
    num_epochs_per_task=5,
    learning_rate=1e-4,
    mc_samples_train=1,
    mc_samples_val=2
)
print("\n Success! The pipeline runs correctly.")

loading annotations into memory...
Done (t=0.36s)
creating index...
index created!
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
Starting POSE ESTIMATION training pipeline...

Task 1/1: POSE


Epoch 1/5 | Loss: 13222.5403 | Val: -858.2088
 Saved best pose model: -858.2088


Epoch 2/5 | Loss: 10684.9999 | Val: -838.4812
 Saved best pose model: -838.4812


Epoch 3/5 | Loss: 10326.6788 | Val: -830.3364
 Saved best pose model: -830.3364


Epoch 4/5 | Loss: 10208.5455 | Val: -827.1452
 Saved best pose model: -827.1452


Epoch 5/5 | Loss: 10204.6423 | Val: -823.2858
 Saved best pose model: -823.2858

 Success! The pipeline runs correctly.


In [18]:
# Test uncertainty output
print("\nTesting uncertainty output...")
model.eval()
val_loader = task_dataloaders['pose']['val']
images, targets = next(iter(val_loader))

# 2. Safely convert the tuple of images into a tensor and move to GPU
if isinstance(images, tuple) or isinstance(images, list):
    _input = torch.stack(images).to(trainer.device)
else:
    _input = images.to(trainer.device)

# Test with single sample (no uncertainty)
output_single = model(_input, num_mc_samples=1, return_uncertainty=True)
print(f" Single forward pass:")
print(f"  - Predictions shape: {output_single['predictions'].shape}")
print(f"  - Uncertainty shape: {output_single['uncertainty'].shape}")
print(f"  - Uncertainty (should be mostly zeros): {output_single['uncertainty'].mean().item():.6f}")

# Test with multiple samples (with uncertainty)
output_multi = model(_input, num_mc_samples=5, return_uncertainty=True)
print(f"\n Multiple MC forward passes (5 samples):")
print(f"  - Predictions shape: {output_multi['predictions'].shape}")
print(f"  - Uncertainty shape: {output_multi['uncertainty'].shape}")
print(f"  - Uncertainty (should be non-zero): {output_multi['uncertainty'].mean().item():.6f}")




Testing uncertainty output...
 Single forward pass:
  - Predictions shape: torch.Size([4, 34])
  - Uncertainty shape: torch.Size([4, 34])
  - Uncertainty (should be mostly zeros): 0.000000

 Multiple MC forward passes (5 samples):
  - Predictions shape: torch.Size([4, 34])
  - Uncertainty shape: torch.Size([4, 34])
  - Uncertainty (should be non-zero): 26.998909


In [19]:
import torch
import torch.nn as nn
import numpy as np

print("="*70)
print("COMPREHENSIVE REAL DATA INPUT & OUTPUT DEMONSTRATION (POSE DETECTION)")
print("="*70)

# 1. Fetch REAL data from your validation dataloader
print("Fetching a batch of real images from the detection dataloader...")
val_loader = task_dataloaders['pose']['val']
images, targets = next(iter(val_loader))

# Determine the device the model is currently on
device = next(model.parameters()).device

# Safely handle tuple/list of images (standard for object detection) and move to GPU
if isinstance(images, (tuple, list)):
    real_input = torch.stack(images).to(device)
else:
    real_input = images.to(device)

print(f"Real Input shape: {real_input.shape} (batch_size, channels, height, width)")

# Set model to evaluation mode and configure for detection
model.eval()
model.set_task('pose')

# =====================================================================
# Test 1: SINGLE FORWARD PASS (No Uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 1: SINGLE FORWARD PASS (num_mc_samples=1)")
print("-"*70)

with torch.no_grad():
    output_single = model(real_input, num_mc_samples=1, return_uncertainty=False)

print(f"Output predictions shape: {output_single['predictions'].shape}")
print(f"  - First 4 values: bounding boxes")
print(f"  - Remaining values: class logits")

print(f"\nDetection Output for Sample 1:")
# Note: Using .cpu() before .numpy() ensures it works even if data is on the GPU
if output_single['predictions'].dim() == 2:
    # Shape: [batch_size, features]
    bbox = output_single['predictions'][0, :4]
    classes = output_single['predictions'][0, 4:]
else:
    # Shape: [batch_size, num_anchors, features]
    bbox = output_single['predictions'][0, 0, :4]
    classes = output_single['predictions'][0, 0, 4:]

print(f"  Bounding Box: {bbox.cpu().detach().numpy()}")
print(f"  Class Logits (first 5): {classes[:5].cpu().detach().numpy()}...")


# =====================================================================
# Test 2: MULTIPLE FORWARD PASSES (With Uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 2: MULTIPLE MC PASSES (num_mc_samples=10) WITH UNCERTAINTY")
print("-"*70)

with torch.no_grad():
    output_multi = model(real_input, num_mc_samples=10, return_uncertainty=True)

print(f"Mean Predictions shape: {output_multi['predictions'].shape}")
print(f"Uncertainty shape: {output_multi['uncertainty'].shape}")

print(f"\nUncertainty for Sample 1:")
if output_multi['uncertainty'].dim() == 2:
    unc_bbox = output_multi['uncertainty'][0, :4]
    unc_classes = output_multi['uncertainty'][0, 4:]
else:
    unc_bbox = output_multi['uncertainty'][0, 0, :4]
    unc_classes = output_multi['uncertainty'][0, 0, 4:]

print(f"  BBox Uncertainty: {unc_bbox.cpu().detach().numpy()}")
print(f"  Class Uncertainty (first 5): {unc_classes[:5].cpu().detach().numpy()}...")
print(f"  Mean overall uncertainty for Image 1: {output_multi['uncertainty'][0].mean().item():.6f}")


# =====================================================================
# Test 3: DETAILED MC INFERENCE (predict_with_uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 3: DETAILED MC INFERENCE (T=20)")
print("-"*70)

with torch.no_grad():
    # Take just the first real image from the batch
    sample = real_input[:1]  
    
    # Run the custom B_YOLO method
    mc_output = model.predict_with_uncertainty(sample, T=20)

print(f"Using T=20 MC forward passes on 1 real sample:")
# print(f"Uncertainty shape: {mc_output['uncertainty'].shape}")
print(f"mc_output : {mc_output}")

if 'confidence' in mc_output:
    print(f"Confidence shape: {mc_output['confidence'].shape}")

print("\n" + "="*70)
print(" All pose detection tests with REAL DATA completed successfully!")
print("="*70)

COMPREHENSIVE REAL DATA INPUT & OUTPUT DEMONSTRATION (POSE DETECTION)
Fetching a batch of real images from the detection dataloader...
Real Input shape: torch.Size([4, 3, 640, 640]) (batch_size, channels, height, width)

----------------------------------------------------------------------
TEST 1: SINGLE FORWARD PASS (num_mc_samples=1)
----------------------------------------------------------------------
Output predictions shape: torch.Size([4, 34])
  - First 4 values: bounding boxes
  - Remaining values: class logits

Detection Output for Sample 1:
  Bounding Box: [     181.77      106.96      160.69      87.042]
  Class Logits (first 5): [     148.06      87.677       148.8      82.025      133.05]...

----------------------------------------------------------------------
TEST 2: MULTIPLE MC PASSES (num_mc_samples=10) WITH UNCERTAINTY
----------------------------------------------------------------------
Mean Predictions shape: torch.Size([4, 34])
Uncertainty shape: torch.Size([4, 

## Segementation

In [9]:
# import torch
# import json
# import torch.nn as nn
# from PIL import Image
# from torch.utils.data import DataLoader,Dataset
# from torchvision.datasets import CocoDetection
# from torchvision.transforms import v2
# from ultralytics import YOLO

# class COCOSegmentationDataset(Dataset):
#     def __init__(self, image_dir, json_path, input_size=(640, 640)):
#         self.image_dir = image_dir
#         self.input_size = input_size # (height, width)
        
#         # Load the COCO JSON
#         with open(json_path, 'r') as f:
#             self.coco = json.load(f)
            
#         # Create quick lookups mapping image IDs to their data and annotations
#         self.images = {img['id']: img for img in self.coco['images']}
#         self.annotations = {}
#         for ann in self.coco['annotations']:
#             img_id = ann['image_id']
#             if img_id not in self.annotations:
#                 self.annotations[img_id] = []
#             self.annotations[img_id].append(ann)
            
#         self.image_ids = list(self.images.keys())
        
#         # Transforms for the input image
#         self.transforms = v2.Compose([
#             v2.ToImage(),
#             v2.Resize(input_size),
#             v2.ToDtype(torch.float32, scale=True),
#         ])

#     def __len__(self):
#         return len(self.image_ids)

#     def __getitem__(self, idx):
#         img_id = self.image_ids[idx]
#         img_info = self.images[img_id]
        
#         # 1. Load the Image
#         img_path = os.path.join(self.image_dir, img_info['file_name'])
#         image = Image.open(img_path).convert("RGB")
#         orig_w, orig_h = image.size
        
#         # Apply standard image transforms -> Shape: [3, 640, 640]
#         image_tensor = self.transforms(image)
        
#         # 2. Create the Mask
#         # Start with a black canvas of the original image size
#         mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
        
#         # Draw the COCO polygons onto the canvas in white (1)
#         anns = self.annotations.get(img_id, [])
#         for ann in anns:
#             if 'segmentation' in ann and isinstance(ann['segmentation'], list):
#                 for poly in ann['segmentation']:
#                     # COCO polygons are flat arrays: [x1, y1, x2, y2...]. Reshape them.
#                     poly_pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
#                     cv2.fillPoly(mask, [poly_pts], 1)
                    
#         # Resize mask to match the model's expected input size (e.g., 640x640)
#         # We use INTER_NEAREST so the values stay strictly 0 or 1
#         mask = cv2.resize(mask, (self.input_size[1], self.input_size[0]), interpolation=cv2.INTER_NEAREST)
        
#         # Convert mask to PyTorch Tensor and add the channel dimension -> Shape: [1, 640, 640]
#         mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        
#         # Return the image and the completed mask!
#         return image_tensor, mask_tensor

# def create_segmentation_loader(image_dir, annotation_json, batch_size=4, input_size=(640, 640)):
#     dataset = COCOSegmentationDataset(image_dir, annotation_json, input_size)
    
#     # Notice we DO NOT need a custom collate_fn! 
#     # PyTorch natively stacks the tensors we generated above.
#     return DataLoader(
#         dataset,
#         batch_size=batch_size,
#         shuffle=True if 'train' in annotation_json.lower() else False,
#         num_workers=4,
#         pin_memory=True
#     )

# # 3. Initialize Model
# backbone = YOLO11sBackbone(pretrained=True)

# # Assuming 1 class (Person) for Pose Estimation
# model = B_YOLO(backbone, num_classes=1) 

# task_dataloaders = {
#     'segmentation': {
#         'train': create_segmentation_loader(
#             image_dir='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/2_Instance_Segmentation/images', 
#             annotation_json='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/2_Instance_Segmentation/train.json', 
#             batch_size=4
#         ),
#         'val': create_segmentation_loader(
#             image_dir='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/2_Instance_Segmentation/images', 
#             annotation_json='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/2_Instance_Segmentation/test.json', 
#             batch_size=4
#         )
#     }
# }

# # Ensure the trainer is set to 'pose'
# trainer = B_YOLOTrainer(model, tasks=['segmentation'])

# print("Starting segmentation training pipeline...")
# trainer.train_sequential_tasks(
#     task_dataloaders,
#     num_epochs_per_task=5,
#     learning_rate=1e-4,
#     mc_samples_train=1,
#     mc_samples_val=2
# )
# print("\n Success! The pipeline runs correctly.")

import os
import json
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from ultralytics import YOLO

# 1. ADD THE MISSING BACKBONE CLASS BACK IN
class YOLO11sBackbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        model_name = "yolo11s-seg.pt" if pretrained else "yolo11s-seg.yaml"
        yolo = YOLO(model_name)
        self.backbone = yolo.model.model[:10]
        
    def forward(self, x):
        for layer in self.backbone:
            x = layer(x)
        return x

# 2. YOUR SEGMENTATION DATASET
class COCOSegmentationDataset(Dataset):
    def __init__(self, image_dir, json_path, input_size=(640, 640)):
        self.image_dir = image_dir
        self.input_size = input_size 
        
        with open(json_path, 'r') as f:
            self.coco = json.load(f)
            
        self.images = {img['id']: img for img in self.coco['images']}
        self.annotations = {}
        for ann in self.coco['annotations']:
            img_id = ann['image_id']
            if img_id not in self.annotations:
                self.annotations[img_id] = []
            self.annotations[img_id].append(ann)
            
        self.image_ids = list(self.images.keys())
        
        self.transforms = v2.Compose([
            v2.ToImage(),
            v2.Resize(input_size),
            v2.ToDtype(torch.float32, scale=True),
        ])

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]
        
        img_path = os.path.join(self.image_dir, img_info['file_name'])
        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size
        
        image_tensor = self.transforms(image)
        
        mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
        
        anns = self.annotations.get(img_id, [])
        for ann in anns:
            if 'segmentation' in ann and isinstance(ann['segmentation'], list):
                for poly in ann['segmentation']:
                    poly_pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
                    cv2.fillPoly(mask, [poly_pts], 1)
                    
        mask = cv2.resize(mask, (self.input_size[1], self.input_size[0]), interpolation=cv2.INTER_NEAREST)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image_tensor, mask_tensor

def create_segmentation_loader(image_dir, annotation_json, batch_size=4, input_size=(640, 640)):
    dataset = COCOSegmentationDataset(image_dir, annotation_json, input_size)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True if 'train' in annotation_json.lower() else False,
        num_workers=4,
        pin_memory=True
    )

# 3. PIPELINE INITIALIZATION
backbone = YOLO11sBackbone(pretrained=True)

# Note: Ensure B_YOLO and B_YOLOTrainer are imported or defined above this in your actual script!
model = B_YOLO(backbone, num_classes=1) 

task_dataloaders = {
    'segmentation': {
        'train': create_segmentation_loader(
            image_dir='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/2_Instance_Segmentation/images', 
            annotation_json='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/2_Instance_Segmentation/train.json', 
            batch_size=4
        ),
        'val': create_segmentation_loader(
            image_dir='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/2_Instance_Segmentation/images', 
            annotation_json='/home/kunal-pg/VLA_0/vla0/Bayesian/BNN/2_Instance_Segmentation/test.json', 
            batch_size=4
        )
    }
}

trainer = B_YOLOTrainer(model, tasks=['segmentation'])

print("Starting segmentation training pipeline...")
trainer.train_sequential_tasks(
    task_dataloaders,
    num_epochs_per_task=10,
    learning_rate=1e-4,
    mc_samples_train=1,
    mc_samples_val=2
)
print("\n Success! The pipeline runs correctly.")

Starting segmentation training pipeline...

Task 1/1: SEGMENTATION


Epoch 1/10 | Loss: 1.3045 | Val: 0.2329
 Saved best segmentation model: 0.2329


Epoch 2/10 | Loss: 1.2114 | Val: 0.2477
 Saved best segmentation model: 0.2477


Epoch 3/10 | Loss: 1.1264 | Val: 0.2524
 Saved best segmentation model: 0.2524


Epoch 4/10 | Loss: 1.0455 | Val: 0.2409


...

Epoch 6/10 | Loss: 0.8869 | Val: 0.2408


Epoch 7/10 | Loss: 0.8111 | Val: 0.2543
 Saved best segmentation model: 0.2543


Epoch 8/10 | Loss: 0.7415 | Val: 0.2425


Epoch 9/10 | Loss: 0.6758 | Val: 0.2480


Epoch 10/10 | Loss: 0.6197 | Val: 0.2480

 Success! The pipeline runs correctly.


In [10]:
# Test uncertainty output
print("\nTesting uncertainty output...")
model.eval()
val_loader = task_dataloaders['segmentation']['val']
images, targets = next(iter(val_loader))

# 2. Safely convert the tuple of images into a tensor and move to GPU
if isinstance(images, tuple) or isinstance(images, list):
    _input = torch.stack(images).to(trainer.device)
else:
    _input = images.to(trainer.device)

# Test with single sample (no uncertainty)
output_single = model(_input, num_mc_samples=1, return_uncertainty=True)
print(f" Single forward pass:")
print(f"  - Predictions shape: {output_single['predictions'].shape}")
print(f"  - Uncertainty shape: {output_single['uncertainty'].shape}")
print(f"  - Uncertainty (should be mostly zeros): {output_single['uncertainty'].mean().item():.6f}")

# Test with multiple samples (with uncertainty)
output_multi = model(_input, num_mc_samples=5, return_uncertainty=True)
print(f"\n Multiple MC forward passes (5 samples):")
print(f"  - Predictions shape: {output_multi['predictions'].shape}")
print(f"  - Uncertainty shape: {output_multi['uncertainty'].shape}")
print(f"  - Uncertainty (should be non-zero): {output_multi['uncertainty'].mean().item():.6f}")




Testing uncertainty output...
 Single forward pass:
  - Predictions shape: torch.Size([4, 1, 640, 640])
  - Uncertainty shape: torch.Size([4, 1, 640, 640])
  - Uncertainty (should be mostly zeros): 0.000000

 Multiple MC forward passes (5 samples):
  - Predictions shape: torch.Size([4, 1, 640, 640])
  - Uncertainty shape: torch.Size([4, 1, 640, 640])
  - Uncertainty (should be non-zero): 0.455402


In [11]:
import torch
import torch.nn as nn
import numpy as np

print("="*70)
print("COMPREHENSIVE REAL DATA INPUT & OUTPUT DEMONSTRATION (SEGMENTATION)")
print("="*70)

# 1. Fetch REAL data from your validation dataloader
print("Fetching a batch of real images from the detection dataloader...")
val_loader = task_dataloaders['segmentation']['val']
images, targets = next(iter(val_loader))

# Determine the device the model is currently on
device = next(model.parameters()).device

# Safely handle tuple/list of images (standard for object detection) and move to GPU
if isinstance(images, (tuple, list)):
    real_input = torch.stack(images).to(device)
else:
    real_input = images.to(device)

print(f"Real Input shape: {real_input.shape} (batch_size, channels, height, width)")

# Set model to evaluation mode and configure for detection
model.eval()
model.set_task('pose')

# =====================================================================
# Test 1: SINGLE FORWARD PASS (No Uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 1: SINGLE FORWARD PASS (num_mc_samples=1)")
print("-"*70)

with torch.no_grad():
    output_single = model(real_input, num_mc_samples=1, return_uncertainty=False)

print(f"Output predictions shape: {output_single['predictions'].shape}")
print(f"  - First 4 values: bounding boxes")
print(f"  - Remaining values: class logits")

print(f"\nDetection Output for Sample 1:")
# Note: Using .cpu() before .numpy() ensures it works even if data is on the GPU
if output_single['predictions'].dim() == 2:
    # Shape: [batch_size, features]
    bbox = output_single['predictions'][0, :4]
    classes = output_single['predictions'][0, 4:]
else:
    # Shape: [batch_size, num_anchors, features]
    bbox = output_single['predictions'][0, 0, :4]
    classes = output_single['predictions'][0, 0, 4:]

print(f"  Bounding Box: {bbox.cpu().detach().numpy()}")
print(f"  Class Logits (first 5): {classes[:5].cpu().detach().numpy()}...")


# =====================================================================
# Test 2: MULTIPLE FORWARD PASSES (With Uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 2: MULTIPLE MC PASSES (num_mc_samples=10) WITH UNCERTAINTY")
print("-"*70)

with torch.no_grad():
    output_multi = model(real_input, num_mc_samples=10, return_uncertainty=True)

print(f"Mean Predictions shape: {output_multi['predictions'].shape}")
print(f"Uncertainty shape: {output_multi['uncertainty'].shape}")

print(f"\nUncertainty for Sample 1:")
if output_multi['uncertainty'].dim() == 2:
    unc_bbox = output_multi['uncertainty'][0, :4]
    unc_classes = output_multi['uncertainty'][0, 4:]
else:
    unc_bbox = output_multi['uncertainty'][0, 0, :4]
    unc_classes = output_multi['uncertainty'][0, 0, 4:]

print(f"  BBox Uncertainty: {unc_bbox.cpu().detach().numpy()}")
print(f"  Class Uncertainty (first 5): {unc_classes[:5].cpu().detach().numpy()}...")
print(f"  Mean overall uncertainty for Image 1: {output_multi['uncertainty'][0].mean().item():.6f}")


# =====================================================================
# Test 3: DETAILED MC INFERENCE (predict_with_uncertainty)
# =====================================================================
print("\n" + "-"*70)
print("TEST 3: DETAILED MC INFERENCE (T=20)")
print("-"*70)

with torch.no_grad():
    # Take just the first real image from the batch
    sample = real_input[:1]  
    
    # Run the custom B_YOLO method
    mc_output = model.predict_with_uncertainty(sample, T=20)

print(f"Using T=20 MC forward passes on 1 real sample:")
# print(f"Uncertainty shape: {mc_output['uncertainty'].shape}")
print(f"mc_output : {mc_output}")

if 'confidence' in mc_output:
    print(f"Confidence shape: {mc_output['confidence'].shape}")

print("\n" + "="*70)
print(" All segmentation tests with REAL DATA completed successfully!")
print("="*70)

COMPREHENSIVE REAL DATA INPUT & OUTPUT DEMONSTRATION (SEGMENTATION)
Fetching a batch of real images from the detection dataloader...
Real Input shape: torch.Size([4, 3, 640, 640]) (batch_size, channels, height, width)

----------------------------------------------------------------------
TEST 1: SINGLE FORWARD PASS (num_mc_samples=1)
----------------------------------------------------------------------
Output predictions shape: torch.Size([4, 34])
  - First 4 values: bounding boxes
  - Remaining values: class logits

Detection Output for Sample 1:
  Bounding Box: [    -26.384       6.258     -3.1055      10.752]
  Class Logits (first 5): [     12.479     -4.9236     -10.861      10.126     0.80296]...

----------------------------------------------------------------------
TEST 2: MULTIPLE MC PASSES (num_mc_samples=10) WITH UNCERTAINTY
----------------------------------------------------------------------
Mean Predictions shape: torch.Size([4, 34])
Uncertainty shape: torch.Size([4, 34

In [6]:
# ============================================================
# SIMPLE DUMMY INPUT → MODEL OUTPUT DEMONSTRATION
# ============================================================

print("\n" + "="*80)
print("SIMPLE MODEL INPUT/OUTPUT EXAMPLE")
print("="*80)

# Initialize model
backbone = DummyBackbone()
model = B_YOLO(backbone, num_classes=5)
model.eval()

# ============================================================
# EXAMPLE 1: CLASSIFICATION TASK
# ============================================================
print("\n" + "-"*80)
print("EXAMPLE 1: CLASSIFICATION TASK")
print("-"*80)

model.set_task('classification')

# Create 2 dummy images
dummy_batch = torch.randn(2, 3, 224, 224)
print(f"\n INPUT: Created dummy batch")
print(f"  Shape: {dummy_batch.shape}")
print(f"  (batch_size=2, channels=3, height=224, width=224)")

# Get single prediction
with torch.no_grad():
    output = model(dummy_batch, num_mc_samples=1, return_uncertainty=False)

print(f"\n OUTPUT: Model predictions")
print(f"  Raw logits shape: {output['predictions'].shape}")
print(f"  Raw logits for image 1: {output['predictions'][0].numpy()}")
print(f"  Raw logits for image 2: {output['predictions'][1].numpy()}")

# Convert to probabilities
probs = torch.softmax(output['predictions'], dim=1)
print(f"\n PROBABILITIES (softmax):")
for i, prob in enumerate(probs):
    pred_class = prob.argmax().item()
    pred_conf = prob.max().item()
    print(f"  Image {i+1}:")
    print(f"    All class probs: {prob.numpy()}")
    print(f"    Predicted class: {pred_class}, Confidence: {pred_conf:.4f}")

# ============================================================
# EXAMPLE 2: CLASSIFICATION WITH UNCERTAINTY
# ============================================================
print("\n" + "-"*80)
print("EXAMPLE 2: CLASSIFICATION WITH UNCERTAINTY (7 MC samples)")
print("-"*80)

with torch.no_grad():
    output = model(dummy_batch, num_mc_samples=7, return_uncertainty=True)

print(f"\n OUTPUT with Uncertainty:")
print(f"  Predictions shape: {output['predictions'].shape}")
print(f"  Uncertainty shape: {output['uncertainty'].shape}")

probs_mean = torch.softmax(output['predictions'], dim=1)
print(f"\n MEAN PREDICTIONS:")
for i, prob in enumerate(probs_mean):
    print(f"  Image {i+1}: {prob.numpy()}")

print(f"\n UNCERTAINTY SCORES:")
for i, unc in enumerate(output['uncertainty']):
    print(f"  Image {i+1}: {unc.numpy()}")
    print(f"           Average uncertainty: {unc.mean().item():.6f}")

# ============================================================
# EXAMPLE 3: DETECTION TASK
# ============================================================
print("\n" + "-"*80)
print("EXAMPLE 3: DETECTION TASK (with 3 MC samples)")
print("-"*80)

model.set_task('detection')

with torch.no_grad():
    output = model(dummy_batch, num_mc_samples=3, return_uncertainty=True)

print(f"\n OUTPUT: Detection predictions")
print(f"  Predictions shape: {output['predictions'].shape}")
print(f"  (2 images, 4 bbox values + 5 class logits = 9 total)")

pred = output['predictions']
print(f"\n IMAGE 1 DETECTION OUTPUT:")
print(f"  Bounding Box [x1, y1, x2, y2]: {pred[0, :4].numpy()}")
print(f"  Class Logits: {pred[0, 4:].numpy()}")

# Convert to probabilities
class_probs = torch.softmax(pred[:, 4:], dim=1)
print(f"\n IMAGE 1 CLASS PROBABILITIES:")
print(f"  Probabilities: {class_probs[0].numpy()}")
print(f"  Predicted class: {class_probs[0].argmax().item()}")

print(f"\n IMAGE 1 UNCERTAINTY:")
unc = output['uncertainty']
print(f"  BBox uncertainty [x1, y1, x2, y2]: {unc[0, :4].numpy()}")
print(f"  Class uncertainty: {unc[0, 4:].numpy()}")
print(f"  Mean uncertainty: {unc[0].mean().item():.6f}")

# ============================================================
# EXAMPLE 4: POSE ESTIMATION TASK
# ============================================================
print("\n" + "-"*80)
print("EXAMPLE 4: POSE ESTIMATION TASK")
print("-"*80)

model.set_task('pose')

with torch.no_grad():
    output = model(dummy_batch, num_mc_samples=1, return_uncertainty=False)

print(f"\n OUTPUT: Pose predictions")
print(f"  Output shape: {output['predictions'].shape}")
print(f"  (2 images, 17 keypoints × 2 coordinates = 34 values)")

pose_pred = output['predictions']
print(f"\n IMAGE 1 POSE KEYPOINTS:")
print(f"  All keypoints (x, y coords): {pose_pred[0].numpy()}")
print(f"  First keypoint (x, y): [{pose_pred[0, 0].item():.4f}, {pose_pred[0, 1].item():.4f}]")
print(f"  Second keypoint (x, y): [{pose_pred[0, 2].item():.4f}, {pose_pred[0, 3].item():.4f}]")

print("\n" + "="*80)
print(" ALL EXAMPLES COMPLETED -  NOW HAVE MODEL OUTPUTS that too with uncertainity scores @ Group 2!")
print("="*80)


SIMPLE MODEL INPUT/OUTPUT EXAMPLE

--------------------------------------------------------------------------------
EXAMPLE 1: CLASSIFICATION TASK
--------------------------------------------------------------------------------

 INPUT: Created dummy batch
  Shape: torch.Size([2, 3, 224, 224])
  (batch_size=2, channels=3, height=224, width=224)

 OUTPUT: Model predictions
  Raw logits shape: torch.Size([2, 5])
  Raw logits for image 1: [ 0.24524617 -0.88951504 -0.02101544 -0.3653097   0.33586183]
  Raw logits for image 2: [ 0.25309828 -0.88582015 -0.02270614 -0.37001783  0.32895842]

 PROBABILITIES (softmax):
  Image 1:
    All class probs: [0.26841065 0.08629373 0.2056666  0.14576012 0.2938689 ]
    Predicted class: 4, Confidence: 0.2939
  Image 2:
    All class probs: [0.27069396 0.08666676 0.20544624 0.14516525 0.2920278 ]
    Predicted class: 4, Confidence: 0.2920

--------------------------------------------------------------------------------
EXAMPLE 2: CLASSIFICATION WITH UNCER

Model Output Summary:

TEST 1: CLASSIFICATION (single forward pass)
Input: 2 dummy images (2×3×224×224)
Output: 5 class predictions (logits)
Image 1: [-0.392, 0.081, 0.062, -0.743, -0.379]
Image 2: [-0.392, 0.080, 0.062, -0.744, -0.384]
Probabilities:
Image 1: [0.170, 0.272, 0.267, 0.119, 0.172] (class 1 has highest confidence ~0.27)

TEST 2: CLASSIFICATION WITH UNCERTAINTY (5 MC samples)
Mean predictions: [-0.412, 0.785, -0.145, 0.374, 0.270]
Uncertainty scores: [0.035, 0.110, 0.092, 0.185, 0.069]
Highest uncertainty in class 3 (0.185)
Lowest uncertainty in class 0 (0.035)

TEST 3: DETECTION OUTPUT
Bounding Box: [-0.581, 0.535, 0.261, -0.486] (x1, y1, x2, y2)
Class Logits: [0.979, 0.403, 0.748, -0.692, -0.208]
BBox Uncertainty: [0.022, 0.271, 0.173, 0.005]
Class Uncertainty: [0.075, 0.065, 0.102, 0.325, 0.110]

In [54]:
# =========================================================
# EVALUATION METRICS UTILITIES
# =========================================================
import torch
import numpy as np

def compute_iou(pred_box, target_box):
    """Computes Intersection over Union (IoU) for bounding boxes."""
    x1 = torch.max(pred_box[:, 0], target_box[:, 0])
    y1 = torch.max(pred_box[:, 1], target_box[:, 1])
    x2 = torch.min(pred_box[:, 2], target_box[:, 2])
    y2 = torch.min(pred_box[:, 3], target_box[:, 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area_p = (pred_box[:, 2] - pred_box[:, 0]) * (pred_box[:, 3] - pred_box[:, 1])
    area_t = (target_box[:, 2] - target_box[:, 0]) * (target_box[:, 3] - target_box[:, 1])

    union = area_p + area_t - inter
    return inter / (union + 1e-8)

def compute_mask_iou(pred_mask, target_mask, threshold=0.5):
    """Computes IoU for flattened segmentation masks."""
    pred_bin = (torch.sigmoid(pred_mask) > threshold).float()
    target_bin = (target_mask > threshold).float()

    intersection = (pred_bin * target_bin).sum(dim=1)
    union = pred_bin.sum(dim=1) + target_bin.sum(dim=1) - intersection
    return intersection / (union + 1e-8)

def compute_oks(pred_pose, target_pose, image_area=224*224):
    """
    Approximates Object Keypoint Similarity (OKS) for Pose Estimation.
    pred_pose, target_pose: (B, 17*2)
    """
    pred_pts = pred_pose.view(-1, 17, 2)
    tgt_pts = target_pose.view(-1, 17, 2)

    # Euclidean distance between keypoints
    distances = torch.norm(pred_pts - tgt_pts, dim=2) ** 2

    # Standard deviation constant for keypoints (simplified)
    k = 0.05
    oks = torch.exp(-distances / (2 * image_area * (k ** 2) + 1e-8))
    return oks.mean(dim=1) # Mean OKS per image

def compute_oriented_iou(pred_box, pred_angle, target_box, target_angle):
    """
    Approximates Rotated IoU by penalizing standard IoU based on angle difference.
    (A true polygon intersection is complex; this serves as a differentiable proxy).
    """
    base_iou = compute_iou(pred_box, target_box)

    # Angle difference penalty (cos of difference)
    angle_diff = torch.abs(pred_angle - target_angle)
    angle_penalty = torch.cos(angle_diff).clamp(min=0)

    return base_iou * angle_penalty

In [ ]:
# =========================================================
# B-YOLO EVALUATOR INFERENCE LOOP
# =========================================================
from tqdm import tqdm
from collections import defaultdict

class B_YOLOEvaluator:
    def __init__(self, model, device=None):
        self.model = model
        # self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def evaluate(self, dataloader, task: str):
        valid_tasks = ['detection', 'segmentation', 'pose', 'oriented', 'classification']
        if task not in valid_tasks:
            raise ValueError(f"Task must be one of {valid_tasks}")

        self.model.set_task(task)
        self.model.eval()

        metrics_tracker = defaultdict(list)

        # IoU thresholds for mAP@50-95
        iou_thresholds = torch.arange(0.5, 1.0, 0.05).to(self.device)

        with torch.no_grad():
            for images, targets in tqdm(dataloader, desc=f"Evaluating {task.upper()}"):
                images = images.to(self.device)
                targets = targets.to(self.device)

                # Standard deterministic inference for evaluation
                outputs = self.model(images, sample=False, return_uncertainty=False)
                preds = outputs['predictions']

                # ---------------------------------------------------------
                # CLASSIFICATION
                # ---------------------------------------------------------
                if task == 'classification':
                    if targets.dim() == 2:
                        targets = targets.argmax(dim=1)

                    pred_classes = preds.argmax(dim=1)
                    top5_preds = preds.topk(min(5, preds.shape[1]), dim=1)[1]

                    metrics_tracker['top1'].extend((pred_classes == targets).tolist())

                    for i in range(targets.size(0)):
                        metrics_tracker['top5'].append(targets[i] in top5_preds[i])

                    # Precision & Recall (Micro-averaged for dummy targets)
                    metrics_tracker['tp'].extend((pred_classes == targets).tolist())

                # ---------------------------------------------------------
                # DETECTION
                # ---------------------------------------------------------
                elif task == 'detection':
                    pred_bbox = preds[:, :4]
                    pred_class = preds[:, 4:].argmax(dim=1)

                    tgt_bbox = targets[:, :4]
                    tgt_class = targets[:, 4:].argmax(dim=1) if targets.dim() == 2 else targets[:, 4]

                    ious = compute_iou(pred_bbox, tgt_bbox)
                    correct_class = (pred_class == tgt_class)

                    # mAP@50
                    tp_50 = (ious >= 0.5) & correct_class
                    metrics_tracker['map_50'].extend(tp_50.tolist())

                    # mAP@50-95
                    for iou_thresh in iou_thresholds:
                        tp_thresh = (ious >= iou_thresh) & correct_class
                        metrics_tracker['map_50_95'].extend(tp_thresh.tolist())

                    metrics_tracker['precision'].extend(correct_class.tolist()) # Class correctness
                    metrics_tracker['recall'].extend((ious >= 0.5).tolist())    # Localization success

                # ---------------------------------------------------------
                # INSTANCE SEGMENTATION
                # ---------------------------------------------------------
                elif task == 'segmentation':
                    mask_ious = compute_mask_iou(preds, targets)

                    # Approximating Box IoU from Masks (dummy conversion for shape matching)
                    box_ious = mask_ious * 0.9 # In a real scenario, extract bounding rects from masks

                    metrics_tracker['mask_map_50'].extend((mask_ious >= 0.5).tolist())
                    metrics_tracker['box_map_50'].extend((box_ious >= 0.5).tolist())

                    for iou_thresh in iou_thresholds:
                        metrics_tracker['mask_map_50_95'].extend((mask_ious >= iou_thresh).tolist())
                        metrics_tracker['box_map_50_95'].extend((box_ious >= iou_thresh).tolist())

                    metrics_tracker['precision'].extend((mask_ious >= 0.5).tolist())
                    metrics_tracker['recall'].extend((mask_ious >= 0.3).tolist()) # Looser threshold for recall

                # ---------------------------------------------------------
                # POSE / KEYPOINT DETECTION
                # ---------------------------------------------------------
                elif task == 'pose':
                    oks_scores = compute_oks(preds, targets)

                    metrics_tracker['pose_map_50'].extend((oks_scores >= 0.5).tolist())

                    for iou_thresh in iou_thresholds:
                        metrics_tracker['pose_map_50_95'].extend((oks_scores >= iou_thresh).tolist())

                    metrics_tracker['precision'].extend((oks_scores >= 0.5).tolist())
                    metrics_tracker['recall'].extend((oks_scores >= 0.3).tolist())

                # ---------------------------------------------------------
                # ORIENTED DETECTION
                # ---------------------------------------------------------
                elif task == 'oriented':
                    pred_bbox = preds[:, :4]
                    pred_angle = preds[:, 4]
                    pred_class = preds[:, 5:].argmax(dim=1)

                    tgt_bbox = targets[:, :4]
                    tgt_angle = targets[:, 4]
                    tgt_class = targets[:, 5:].argmax(dim=1) if targets.dim() == 2 else targets[:, 5]

                    rotated_ious = compute_oriented_iou(pred_bbox, pred_angle, tgt_bbox, tgt_angle)
                    correct_class = (pred_class == tgt_class)

                    tp_50 = (rotated_ious >= 0.5) & correct_class
                    metrics_tracker['obb_map_50'].extend(tp_50.tolist())

                    for iou_thresh in iou_thresholds:
                        tp_thresh = (rotated_ious >= iou_thresh) & correct_class
                        metrics_tracker['obb_map_50_95'].extend(tp_thresh.tolist())

                    metrics_tracker['precision'].extend(correct_class.tolist())
                    metrics_tracker['recall'].extend((rotated_ious >= 0.5).tolist())

        # Compile final dictionary
        results = {}
        for key, values in metrics_tracker.items():
            results[key] = np.mean(values) if values else 0.0

        self._print_results(task, results)
        return results

    def _print_results(self, task, results):
        print(f"\n{'='*50}")
        print(f"EVALUATION RESULTS: {task.upper()}")
        print(f"{'='*50}")

        if task == 'classification':
            print(f" Top-1 Accuracy : {results.get('top1', 0):.4f}")
            print(f" Top-5 Accuracy : {results.get('top5', 0):.4f}")
            print(f" Precision      : {results.get('tp', 0):.4f}")
            print(f" Recall         : {results.get('tp', 0):.4f}")

        elif task == 'detection':
            print(f" mAP@50         : {results.get('map_50', 0):.4f}")
            print(f" mAP@50-95      : {results.get('map_50_95', 0):.4f}")
            print(f" Precision      : {results.get('precision', 0):.4f}")
            print(f" Recall         : {results.get('recall', 0):.4f}")

        elif task == 'segmentation':
            print(f" Mask mAP@50    : {results.get('mask_map_50', 0):.4f}")
            print(f" Mask mAP@50-95 : {results.get('mask_map_50_95', 0):.4f}")
            print(f" Box mAP@50     : {results.get('box_map_50', 0):.4f}")
            print(f" Box mAP@50-95  : {results.get('box_map_50_95', 0):.4f}")
            print(f" Precision      : {results.get('precision', 0):.4f}")
            print(f" Recall         : {results.get('recall', 0):.4f}")

        elif task == 'pose':
            print(f" Pose mAP@50    : {results.get('pose_map_50', 0):.4f}")
            print(f" Pose mAP@50-95 : {results.get('pose_map_50_95', 0):.4f}")
            print(f" Precision      : {results.get('precision', 0):.4f}")
            print(f" Recall         : {results.get('recall', 0):.4f}")

        elif task == 'oriented':
            print(f" OBB mAP@50     : {results.get('obb_map_50', 0):.4f}")
            print(f" OBB mAP@50-95  : {results.get('obb_map_50_95', 0):.4f}")
            print(f" Precision      : {results.get('precision', 0):.4f}")
            print(f" Recall         : {results.get('recall', 0):.4f}")
        print(f"{'='*50}\n")

In [9]:
# =========================================================
# SMOKE TEST: RUNNING THE INFERENCE LOOP
# =========================================================
from torch.utils.data import DataLoader, TensorDataset

# Assuming `model` and `DummyBackbone` from your previous cells exist
evaluator = B_YOLOEvaluator(model)

# 1. Detection Data
det_x = torch.randn(4, 3, 224, 224)
det_y = torch.cat([torch.rand(4, 4), torch.randn(4, 10)], dim=1) # 4 bbox + 10 logits
det_loader = DataLoader(TensorDataset(det_x, det_y), batch_size=2)

# 2. Segmentation Data
seg_x = torch.randn(4, 3, 224, 224)
seg_y = torch.rand(4, 256) # 256 mask values
seg_loader = DataLoader(TensorDataset(seg_x, seg_y), batch_size=2)

# 3. Pose Data
pose_x = torch.randn(4, 3, 224, 224)
pose_y = torch.randn(4, 34) # 17 * 2 keypoints
pose_loader = DataLoader(TensorDataset(pose_x, pose_y), batch_size=2)

# 4. Oriented Data
ori_x = torch.randn(4, 3, 224, 224)
ori_y = torch.cat([torch.rand(4, 4), torch.rand(4, 1)*3.14, torch.randn(4, 10)], dim=1) # 4 bbox + 1 angle + 10 logits
ori_loader = DataLoader(TensorDataset(ori_x, ori_y), batch_size=2)

# 5. Classification Data
cls_x = torch.randn(4, 3, 224, 224)
cls_y = torch.randint(0, 10, (4,))
cls_loader = DataLoader(TensorDataset(cls_x, cls_y), batch_size=2)

# Run Inference Loops
det_results = evaluator.evaluate(det_loader, 'detection')
seg_results = evaluator.evaluate(seg_loader, 'segmentation')
pose_results = evaluator.evaluate(pose_loader, 'pose')
ori_results = evaluator.evaluate(ori_loader, 'oriented')
cls_results = evaluator.evaluate(cls_loader, 'classification')

Evaluating DETECTION: 100%|██████████| 2/2 [00:00<00:00,  6.19it/s]



EVALUATION RESULTS: DETECTION
 mAP@50         : 0.0000
 mAP@50-95      : 0.0000
 Precision      : 0.0000
 Recall         : 0.0000



Evaluating SEGMENTATION: 100%|██████████| 2/2 [00:00<00:00,  6.60it/s]



EVALUATION RESULTS: SEGMENTATION
 Mask mAP@50    : 0.0000
 Mask mAP@50-95 : 0.0000
 Box mAP@50     : 0.0000
 Box mAP@50-95  : 0.0000
 Precision      : 0.0000
 Recall         : 0.7500



Evaluating POSE: 100%|██████████| 2/2 [00:00<00:00,  6.45it/s]



EVALUATION RESULTS: POSE
 Pose mAP@50    : 1.0000
 Pose mAP@50-95 : 1.0000
 Precision      : 1.0000
 Recall         : 1.0000



Evaluating ORIENTED: 100%|██████████| 2/2 [00:00<00:00,  6.42it/s]



EVALUATION RESULTS: ORIENTED
 OBB mAP@50     : 0.0000
 OBB mAP@50-95  : 0.0000
 Precision      : 0.0000
 Recall         : 0.0000



Evaluating CLASSIFICATION: 100%|██████████| 2/2 [00:00<00:00,  6.46it/s]


EVALUATION RESULTS: CLASSIFICATION
 Top-1 Accuracy : 0.0000
 Top-5 Accuracy : 0.7500
 Precision      : 0.0000
 Recall         : 0.0000

